# One-Phonon Diffuse Scattering — Rigid-Body Crystal (6o2h, P1)

One rigid body per unit cell (the lysozyme molecule), 6 degrees of freedom per
cell: a rotation $\Omega$ and a translation $v$. The crystal's phonons are the
normal modes of a Born–von Kármán lattice of these rigid bodies, coupled through
a small set of pairwise contact springs. This notebook builds the model from the
deposited structure, verifies it end-to-end against **synthetic** data with a
known ground-truth stiffness, and visualizes the resulting phonon modes.

**Pipeline:** atomic model and mass matrix → molecular transform $F,L$ → coupling
vector $G(\mathbf q)$ → contact detection, framing, and the geometric prior →
dynamical matrix → band structure and diffuse-scattering maps → synthetic-data
refinement → sloppiness analysis → predicted atomic displacement parameters →
mode animations.

**What this notebook can and cannot validate.** A synthetic test that generates
data and fits it with the *same* code paths will pass even when those paths are
wrong together. The ADP comparison is the clearest case: comparing $\Sigma_{\rm
true}$ against $\Sigma_{\rm fit}$ cancels any error common to both. So the
notebook now carries explicit **cross-path consistency checks** — the batched
eigendecomposition used for maps and ADPs is asserted against the single-point
linear solve used by the refinement objective, and $G$ is asserted to retain a
non-negligible imaginary part. Those are the checks a synthetic ground truth
cannot supply on its own.


In [ ]:
import io
import pathlib
from itertools import product

import numpy as np
import gemmi
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.lines import Line2D
import scipy.constants as const
from scipy.spatial import cKDTree
from scipy.linalg import eigh
from scipy.ndimage import zoom as nd_zoom
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components as sc_connected_components
from scipy.interpolate import griddata
from scipy.optimize import minimize
from scipy.spatial.transform import Rotation as Rot
import imageio
import imageio.v2 as iio2
from skimage.measure import marching_cubes
from IPython.display import Image as IPImage, display
import gc

np.set_printoptions(precision=4, suppress=True)

def fig_to_image(fig, dpi=72):
    """Rasterize a matplotlib figure to an RGB array, for building GIFs."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', facecolor=fig.get_facecolor())
    buf.seek(0)
    return iio2.imread(buf)[:, :, :3]


In [ ]:
PDB_FILE = pathlib.Path("6o2h.cif")
SF_CIF   = pathlib.Path("6o2h-sf.cif")

D_MIN_MODEL    = 1.5    # Å — resolution for the visualization/reference density
RATE_MODEL     = 1.5
CONTACT_CUTOFF = 4.0    # Å — atom-atom cutoff defining a rigid-body contact
MIN_CONTACTS   = 5      # minimum atom-atom pairs for a contact to count
MAX_IMAGE      = 1      # search ±MAX_IMAGE unit cells (verified sufficient below)
D_MIN_DIFFUSE  = 3.0    # Å — resolution cutoff for the diffuse map
NPLOT_HK       = 8      # sub-integer grid density per Miller index
TEMPERATURE    = 300.0  # K — used only for the frequency-unit conversion

# --- Prior and regularization ---------------------------------------------
# PRIOR_K_PER_PAIR is the stiffness of one atom-pair contact spring, in k_BT/Å².
# 1 k_BT/Å² = 0.414 N/m, a reasonable order of magnitude for a weak non-covalent
# contact (vdW ≈ 1 N/m, H-bond ≈ 10-100 N/m). The prior's *shape* — how it
# scales with the number of atom pairs, and the κ_R/κ_T ratio — is fixed by the
# contact geometry and is the part that matters; this scalar only sets where the
# refinement starts.
PRIOR_K_PER_PAIR = 1.0
LAMBDA_PRIOR     = 1e-2   # weight on ||K - K_prior||²; set from the Gauss-Newton
                          # spectrum (see the sloppiness section)

CHUNK_EVAL = 20000      # q-points per batch in the vectorized evaluators
BZ_NGRID   = 20         # Brillouin-zone grid for the ADP integral (convergence
                        # is checked explicitly, never assumed)

np.set_printoptions(precision=4, suppress=True)

In [ ]:
st = gemmi.read_structure(str(PDB_FILE))
st.remove_hydrogens()
st.remove_waters()
st.setup_entities()

cell = st.cell
a1 = np.array(cell.orthogonalize(gemmi.Fractional(1,0,0)).tolist())
a2 = np.array(cell.orthogonalize(gemmi.Fractional(0,1,0)).tolist())
a3 = np.array(cell.orthogonalize(gemmi.Fractional(0,0,1)).tolist())
A_orth  = np.column_stack([a1, a2, a3])       # orthogonalization matrix
B_recip = 2*np.pi * np.linalg.inv(A_orth).T   # reciprocal lattice (columns)

print(f"Cell: {cell.a:.3f}×{cell.b:.3f}×{cell.c:.3f} Å  "
      f"α={cell.alpha:.2f} β={cell.beta:.2f} γ={cell.gamma:.2f}°")
print(f"Space group: {st.spacegroup_hm}   Volume: {cell.volume:.1f} Å³")


## Atomic Model, Mass Matrix, and the Molecular Transform

Everything downstream is referenced to **one** point: `r_cm_at`, the atomic
centre of mass. That choice is forced, not cosmetic — the mass matrix
$M=\mathrm{diag}(J,\,mI)$ is block-diagonal (no translation–rotation coupling)
only about the centre of mass, and a rigid displacement
$\delta\mathbf r=\mathbf v+\boldsymbol\Omega\times(\mathbf r-\mathbf r_{\rm ref})$
splits into $(\boldsymbol\Omega,\mathbf v)$ differently for every choice of
$\mathbf r_{\rm ref}$. If $G$ and $D$ are built about different reference
points, $I=G^\dagger D^{-1}G$ contracts two incompatible coordinate systems.

**Why the molecular transform is computed atom-by-atom, not from a density
grid.** The crystal density on a periodic grid is
$\rho_{\rm cell}(\mathbf r)=\sum_{\mathbf n}\rho_{\rm mol}(\mathbf r-\mathbf R_{\mathbf n})$
folded into one box, so

$$F_{\rm cell}(\mathbf q)=\sum_{\mathbf n}e^{-i\mathbf q\cdot\mathbf R_{\mathbf n}}
\int_{\rm box-\mathbf R_{\mathbf n}}\rho_{\rm mol}(\mathbf r')e^{-i\mathbf q\cdot\mathbf r'}d^3r'.$$

At **integer** $hkl$ every phase factor is 1 and $F_{\rm cell}=F_{\rm mol}$. At
the **fractional** $hkl$ this whole pipeline runs on, they are not, so
$F_{\rm cell}\neq F_{\rm mol}$ for any molecule that crosses a cell boundary —
and for a 14 kDa protein in a 27x32x34 Å cell there is no roll of the grid that
leaves empty margins, because crystal contacts are contiguous by construction.
Summing over the deposited (unwrapped) coordinates with IT92 form factors is
exact, unambiguous about $\mathbf r_{\rm cm}$, and roughly 300x faster
(1001 atoms vs. 311040 voxels per $\mathbf q$-point).

The experimentally measured Bragg amplitudes are still used, but as a smooth
**resolution-dependent amplitude correction** $\langle|F_{\rm meas}|\rangle/
\langle|F_{\rm calc}|\rangle$ applied to the atomic transform, rather than by
building a hybrid density that would reintroduce the wrapping problem.


In [ ]:
# --- Atomic model: positions, masses, deposited ADPs, mass matrix ----------
# Built FIRST, because r_cm_at is the reference point for G, for D, and for the
# ADP projection alike (see markdown above).
ATOMIC_MASS = {'C':12.011,'N':14.007,'O':15.999,'S':32.06,'P':30.974,
               'SE':78.96,'H':1.008,'FE':55.845,'ZN':65.38,'CA':40.078}

masses, apos, b_exp, u_exp, has_aniso, elements = [], [], [], [], [], []
for ch in st[0]:
    for res in ch:
        for atom in res:
            masses.append(ATOMIC_MASS.get(atom.element.name.upper(), 12.0))
            apos.append(atom.pos.tolist())
            b_exp.append(atom.b_iso)
            u_exp.append(atom.aniso.as_mat33().tolist())
            has_aniso.append(atom.aniso.nonzero())
            elements.append(atom.element)
masses = np.array(masses); apos = np.array(apos); b_exp = np.array(b_exp)
u_exp = np.array(u_exp); has_aniso = np.array(has_aniso)
all_pos = apos                      # same array, used by the contact search below
n_atoms = len(apos)

m_total = masses.sum()
r_cm_at = (masses[:, None]*apos).sum(0)/m_total     # THE reference point
dr_a    = apos - r_cm_at
r2      = (dr_a**2).sum(1)
J       = (masses[:, None, None]*(r2[:, None, None]*np.eye(3)[None]
          - dr_a[:, :, None]*dr_a[:, None, :])).sum(0)
M_mat   = np.block([[J, np.zeros((3,3))], [np.zeros((3,3)), m_total*np.eye(3)]])
Msq     = np.linalg.cholesky(M_mat)
Msq_inv = np.linalg.inv(Msq)

kBT_SI     = const.k * TEMPERATURE
omega_unit = np.sqrt(kBT_SI / (const.atomic_mass * (1e-10)**2))
freq_unit  = omega_unit / (2*np.pi) / 1e12   # THz per sqrt(reduced stiffness/mass)

print(f"{n_atoms} atoms   total mass {m_total:.0f} amu")
print(f"Centre of mass r_cm_at = ({r_cm_at[0]:.3f}, {r_cm_at[1]:.3f}, {r_cm_at[2]:.3f}) Å")
print(f"Deposited ANISOU records present for {has_aniso.mean():.1%} of atoms")
print(f"Deposited mean B = {b_exp.mean():.2f} Å²")
print(f"Frequency unit: 1 (k_BT/amu/Å²)^½ = {freq_unit:.3f} THz")

# The molecule must not be split across the cell boundary in the deposited
# coordinates either -- check its extent against the cell, since a wrapped
# deposition would break the atomic transform in exactly the same way a
# periodic density grid does.
frac = np.linalg.solve(A_orth, (apos - r_cm_at).T).T
span = frac.max(0) - frac.min(0)
print(f"Molecular extent in fractional coordinates: {span.round(3)} "
      f"({'OK, contiguous' if (span < 1.0).all() else 'WARNING: spans a full cell axis'})")

# --- IT92 form-factor coefficients, one row per atom -----------------------
def _it92_coeffs(el):
    """(a, b, c) for one element, tolerant of gemmi's 4- vs 5-Gaussian layouts."""
    t = el.it92
    a = np.asarray(list(t.a), float)
    b = np.asarray(list(t.b), float)
    c = float(getattr(t, 'c', 0.0))
    return a, b, c

_ab = [_it92_coeffs(el) for el in elements]
_na = max(len(a) for a, b, c in _ab)
FF_A = np.zeros((n_atoms, _na)); FF_B = np.zeros((n_atoms, _na)); FF_C = np.zeros(n_atoms)
for i, (a, b, c) in enumerate(_ab):
    FF_A[i, :len(a)] = a; FF_B[i, :len(b)] = b; FF_C[i] = c
Z_atomic = (FF_A.sum(1) + FF_C).sum()
print(f"Σ f(0) over all atoms = {Z_atomic:.1f} e⁻  (electron count of the model)")


In [ ]:
# --- Model density: used ONLY for the molecular isosurface at the end of the
# notebook, and to derive the measured/calculated amplitude correction below.
# It is deliberately NOT used to compute G(q): see the markdown above for why a
# periodic density grid gives the wrong F(q) at non-integer (h,k,l).
dc = gemmi.DensityCalculatorX()
dc.d_min = D_MIN_MODEL; dc.rate = RATE_MODEL
dc.set_grid_cell_and_spacegroup(st)
dc.initialize_grid()
dc.add_model_density_to_grid(st[0])
dc.grid.symmetrize_sum()
rho_model = np.array(dc.grid)
print(f"Visualization density grid: {rho_model.shape[0]}×{rho_model.shape[1]}×{rho_model.shape[2]}")


In [ ]:
# --- Molecular transform F(q), L(q) by direct atomic summation -------------
#
#   F(q) = Σ_a f_a(|q|) e^{-i q·r_a}            (electrons)
#   L(q) = Σ_a f_a(|q|) (r_a - r_cm_at) e^{-i q·r_a}   (electrons · Å)
#
# USE_ADP_IN_TRANSFORM:
#   True  (default) -- multiply each atom by exp(-B_a s²), giving the
#         Debye-Waller-smeared transform, i.e. the same object the measured
#         Bragg amplitudes represent. This is the conventional choice and keeps
#         the amplitude correction below self-consistent.
#         CAVEAT: the deposited B already contains the lattice motion that
#         D(q)^-1 is also modelling, so this mildly double-counts Debye-Waller
#         at the one-phonon level. The effect is small at the resolutions where
#         this model is valid; set to False to see the size of it.
#   False -- the rigid, unsmeared molecular transform.
USE_ADP_IN_TRANSFORM = True

def _form_factors(s2):
    """f_a(s²) for every atom, s = |q|/4π. Returns (n_q, n_atoms)."""
    f = (FF_A[None, :, :] * np.exp(-FF_B[None, :, :] * s2[:, None, None])).sum(-1) \
        + FF_C[None, :]
    if USE_ADP_IN_TRANSFORM:
        f = f * np.exp(-b_exp[None, :] * s2[:, None])
    return f

# Amplitude correction: replace the *magnitude* of the calculated transform
# with the measured one, as a smooth function of resolution. This is how the
# experimental Bragg amplitudes enter, without building a hybrid density (which
# would reintroduce the unit-cell wrapping error, see markdown above).
APPLY_AMPLITUDE_CORRECTION = True
N_AMP_BINS = 24

def _build_amplitude_correction():
    doc_sf = gemmi.cif.read(str(SF_CIF))
    rb     = gemmi.as_refln_blocks(doc_sf)[0]
    mil    = np.array(rb.make_miller_array())
    Fm     = np.array(rb.make_float_array("F_meas_au"))
    ok     = np.isfinite(Fm) & (Fm > 0)
    mil, Fm = mil[ok], Fm[ok]
    q  = mil @ B_recip.T
    qn = np.linalg.norm(q, axis=1)
    s2 = (qn/(4*np.pi))**2
    f  = _form_factors(s2)
    Fc = np.abs(np.einsum('na,na->n', f, np.exp(-1j*(q @ apos.T))))
    edges = np.quantile(qn, np.linspace(0, 1, N_AMP_BINS+1))
    edges[0] = 0.0; edges[-1] = qn.max()*1.5
    idx = np.clip(np.digitize(qn, edges)-1, 0, N_AMP_BINS-1)
    mid, rat = [], []
    for b in range(N_AMP_BINS):
        m = idx == b
        if m.sum() < 20:
            continue
        mid.append(0.5*(edges[b]+edges[b+1]))
        rat.append(Fm[m].mean()/max(Fc[m].mean(), 1e-12))
    mid, rat = np.array(mid), np.array(rat)
    print(f"Amplitude correction from {len(Fm)} reflections, {len(mid)} usable shells; "
          f"|F_meas|/|F_calc| ranges {rat.min():.3f}–{rat.max():.3f}")
    return mid, rat

if APPLY_AMPLITUDE_CORRECTION:
    _AMP_Q, _AMP_R = _build_amplitude_correction()
    def amplitude_correction(qn):
        return np.interp(qn, _AMP_Q, _AMP_R, left=_AMP_R[0], right=_AMP_R[-1])
else:
    def amplitude_correction(qn):
        return np.ones_like(qn)

def F_L_batch(h_arr, k_arr, l_arr, chunk=4000):
    """F(q) and L(q) at arbitrary fractional Miller indices. Memory is bounded
    by an (chunk × n_atoms) phase matrix -- tiny, since n_atoms ≈ 10³."""
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float)
    l_arr = np.asarray(l_arr, float)
    N = len(h_arr)
    F_out = np.zeros(N, complex); L_out = np.zeros((N, 3), complex)
    d = apos - r_cm_at
    for s in range(0, N, chunk):
        sl = slice(s, min(s+chunk, N))
        q  = (h_arr[sl, None]*B_recip[:, 0] + k_arr[sl, None]*B_recip[:, 1]
              + l_arr[sl, None]*B_recip[:, 2])
        qn = np.linalg.norm(q, axis=1)
        w  = _form_factors((qn/(4*np.pi))**2) * np.exp(-1j*(q @ apos.T))
        w *= amplitude_correction(qn)[:, None]
        F_out[sl] = w.sum(1)
        L_out[sl] = w @ d
    return F_out, L_out

_F0, _L0 = F_L_batch([0.0], [0.0], [0.0])
print(f"F(000) = {_F0[0].real:.1f} e⁻   (Σ f(0) = {Z_atomic:.1f})")
print(f"L(000) = {_L0[0].real.round(2)} e⁻·Å   "
      f"(the electron first moment about the MASS centre; small, not exactly zero)")


## Coupling Vector $G(\mathbf{q})$

$$G = \begin{pmatrix}G_R\\G_T\end{pmatrix}
    = \begin{pmatrix}i\mathbf{q}\times L(\mathbf{q})\\
                      i\mathbf{q}\,F(\mathbf{q})\end{pmatrix},
\quad F = \sum_a f_a e^{-i\mathbf{q}\cdot\mathbf{r}_a},
\quad L = \sum_a f_a(\mathbf{r}_a-\mathbf{r}_{\rm cm})\,e^{-i\mathbf{q}\cdot\mathbf{r}_a}$$

with $\mathbf r_{\rm cm}=$ `r_cm_at`, the same point the dynamical matrix uses.

$G$ is **complex**, and it stays complex everywhere below. $I=G^\dagger D^{-1}G$
is a Hermitian quadratic form; replacing $G$ by $\mathrm{Re}\,G$ (or $D$ by
$\mathrm{Re}\,D$) is not an approximation but a different quantity — on this
geometry it costs a median factor of $\approx 0.46$ in $I(\mathbf q)$, with a
5x spread. Two consistency checks below guard against that class of error.


In [ ]:
def G_at_hkl_batch(h_arr, k_arr, l_arr):
    """Vectorized G(q) at arrays of fractional Miller indices. (N, 6) complex."""
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float)
    l_arr = np.asarray(l_arr, float)
    q  = (h_arr[:, None]*B_recip[:, 0] + k_arr[:, None]*B_recip[:, 1]
          + l_arr[:, None]*B_recip[:, 2])
    iq = 1j*q
    F, L = F_L_batch(h_arr, k_arr, l_arr)
    return np.concatenate([np.cross(iq, L), iq*F[:, None]], axis=1)

def G_at_hkl(h, k, l):
    return G_at_hkl_batch([h], [k], [l])[0]

_Gt = G_at_hkl(1.5, -0.75, 0.25)
print("G at a representative fractional hkl (complex, 6 components):")
print("  |Re G| =", np.abs(_Gt.real).round(1))
print("  |Im G| =", np.abs(_Gt.imag).round(1))
assert np.abs(_Gt.imag).max() > 1e-6*np.abs(_Gt).max(), \
    "G has no imaginary part -- something has silently realified the transform"
print("OK: G retains a substantial imaginary part, as it must.")


## Rigid-Body Contacts

Two unit-cell images are in contact if any of their atoms come within
`CONTACT_CUTOFF`. Born–von Kármán translational symmetry means the contact
between cell 0 and cell $\mathbf n$ depends only on $\mathbf n$, and the two
directions of a contact ($\mathbf n$ and $-\mathbf n$) are the same physical
spring seen from either side — which is why 12 directed images collapse to 6
independent stiffnesses.

Note this is *not* a point-group symmetry reduction: in $P1$ there is no point
symmetry at all, so the six contacts are genuinely **distinct**, not
"symmetry-distinct". They share nothing, and each carries its own free
$6\times6$ matrix.

For each contact we record the centroid of the atom pairs actually within the
cutoff — the physical location of the contact patch — which sets both the
contact-local reference frame and the geometric prior on $K$.


In [ ]:
tree = cKDTree(all_pos)

def canonical(n):
    return min(n, tuple(-x for x in n))

def _find_interfaces(max_image):
    found = {}
    for n_tup in product(range(-max_image, max_image+1), repeat=3):
        if n_tup == (0,0,0):
            continue
        R_n = sum(n_tup[i]*[a1,a2,a3][i] for i in range(3))
        idx_lists = tree.query_ball_point(all_pos + R_n, CONTACT_CUTOFF)
        n_c = sum(len(idx) for idx in idx_lists)
        if n_c >= MIN_CONTACTS:
            found[n_tup] = {'R_n': R_n, 'n_contacts': n_c, 'idx_lists': idx_lists}
    return found

raw_interfaces = _find_interfaces(MAX_IMAGE)

# Verify MAX_IMAGE is actually large enough rather than assuming it. One of the
# detected contacts here sits at |R_n| ≈ 47 Å, so the molecule is long enough
# that second-shell images are worth ruling out explicitly.
_outer = {n: v for n, v in _find_interfaces(MAX_IMAGE+1).items()
          if max(abs(x) for x in n) > MAX_IMAGE}
if _outer:
    print(f"WARNING: {len(_outer)} contact(s) found beyond MAX_IMAGE={MAX_IMAGE}: "
          f"{sorted(_outer)} -- increase MAX_IMAGE.")
else:
    print(f"Checked shell {MAX_IMAGE+1}: no additional contacts, MAX_IMAGE={MAX_IMAGE} is sufficient.")

unique = {}
for n_tup, info in raw_interfaces.items():
    c = canonical(n_tup)
    if c not in unique:
        unique[c] = dict(info, n_tup=n_tup)

def contact_midpoints(R_n, idx_lists):
    """Midpoints (lab frame) of every atom pair within CONTACT_CUTOFF."""
    j_idx = np.concatenate([np.full(len(ii), j, int) for j, ii in enumerate(idx_lists) if len(ii)]) \
            if any(len(ii) for ii in idx_lists) else np.zeros(0, int)
    i_idx = np.concatenate([np.asarray(ii, int) for ii in idx_lists if len(ii)]) \
            if any(len(ii) for ii in idx_lists) else np.zeros(0, int)
    return 0.5*(all_pos[i_idx] + all_pos[j_idx] + R_n)

for c, info in unique.items():
    info['midpoints']  = contact_midpoints(info['R_n'], info['idx_lists'])
    info['contact_pt'] = info['midpoints'].mean(0)

shell_order = sorted(unique.keys())
print(f"\nDetected {len(unique)} distinct contacts ({len(raw_interfaces)} directed images):")
for c in shell_order:
    info = unique[c]
    d = info['midpoints'] - info['contact_pt']
    rg = np.sqrt((d**2).sum(1).mean())
    print(f"  n={c}  |R_n|={np.linalg.norm(info['R_n']):6.2f} Å  "
          f"atom-pairs={info['n_contacts']:3d}  patch gyration radius={rg:5.2f} Å")


## Reduced Units

Stiffnesses are expressed directly in units of $k_BT$, so $k_BT=1$ throughout and
no Boltzmann factor appears in the diffuse-intensity or displacement formulas.

One consequence is worth stating because it is a free correctness test: in the
classical limit the generalized-coordinate covariance is
$\langle u_{\mathbf q}u_{\mathbf q}^\dagger\rangle = k_BT\,D(\mathbf q)^{-1}$,
which contains **no mass matrix at all**. $M$ therefore affects only the plotted
frequencies — never $I(\mathbf q)$, never the ADPs. At room temperature the
classical limit is amply justified here: the branches sit below $\sim0.4$ THz,
i.e. $\hbar\omega\lesssim1.6$ meV against $k_BT=25.9$ meV, so
$\hbar\omega/k_BT\lesssim0.06$.

In [ ]:
# The mass matrix was built alongside the atomic model above (it has to be, since
# r_cm_at is also G's reference point). Report it here and check the classical
# limit's mass-independence explicitly.
print(f"Moment of inertia J (amu·Å²), eigenvalues: {np.linalg.eigvalsh(J).round(0)}")
print(f"Total mass: {m_total:.0f} amu")
print(f"Frequency unit: 1 (k_BT/amu/Å²)^½ = {freq_unit:.3f} THz")

hbar_over_kT_THz = const.hbar*2*np.pi*1e12/(const.k*TEMPERATURE)
print(f"\nClassical-limit check: ħω/k_BT = {hbar_over_kT_THz:.4f} × (ω in THz)")
print(f"  at 0.4 THz -> {0.4*hbar_over_kT_THz:.3f};  at 2 THz -> {2*hbar_over_kT_THz:.3f}")
print("  Both « 1, so the classical (equipartition) form is appropriate.")

## Contact Frames, the Geometric Prior, and the Stiffness Parameterization

Each contact's $6\times6$ stiffness $K$ is specified in a **local contact
frame**: origin at the measured contact centroid, $z$-axis along
$\mathbf R_{\mathbf n}$. A fixed rigid shift-and-rotation (computed once from
the structure, never fit) re-expresses $K$ at the far body's own centre of mass,
which is the representation the dynamical matrix uses.

**The prior.** Model the contact as $n$ independent isotropic point springs of
stiffness $k$ sitting at the atom-pair midpoints $\mathbf m_i$, with
$\mathbf d_i=\mathbf m_i-\bar{\mathbf m}$:

$$E=\tfrac{k}{2}\sum_i\bigl|\mathbf v+\boldsymbol\Omega\times\mathbf d_i\bigr|^2
\;\Longrightarrow\;
K_{TT}=k\,n\,I_3,\qquad
K_{RR}=k\sum_i\bigl(|\mathbf d_i|^2I-\mathbf d_i\mathbf d_i^{\mathsf T}\bigr),\qquad
K_{TR}=0$$

the last exactly, because $\sum_i\mathbf d_i=0$ at the centroid. This costs no
extra parameters and supplies four things a flat isotropic guess cannot:

* **scaling with the number of atom–atom contacts** (9 to 63 here, a 7x spread);
* **the correct $\kappa_R/\kappa_T$ ratio**, which is $\sim\rho_g^2$ with
  $\rho_g$ the patch gyration radius. Setting $\kappa_R=\kappa_T=1$ — one in
  $k_BT/\mathrm{rad}^2$, the other in $k_BT/\mathrm{Å}^2$ — makes the
  librational springs roughly $25$–$60\times$ too soft, and the librational
  branches are exactly the ones that go pathological in an unconstrained fit;
* **the correct patch anisotropy** (a flat contact is automatically soft about
  axes lying in its own plane);
* **a vanishing local-frame $T$–$R$ block**, which is the property an ad-hoc
  regularizer was previously trying to impose by hand.

The refinement is then regularized **toward this prior**, not toward zero. For
a sloppy model that distinction is decisive: penalizing only the $T$–$R$ block
leaves $\sim18$ directions per contact — including every overall magnitude —
with nothing holding them, so they drift to whichever boundary the optimizer
reaches first.

**Stability.** $K=LL^{\mathsf T}$ with $L$ lower-triangular and otherwise free
is positive semidefinite for any real $L$, so the optimizer runs unconstrained.
And that is *sufficient* for the whole crystal, because each contact enters the
dynamical matrix only as

$$D_{\mathbf n}(\mathbf q)=M_{\mathbf n}(\mathbf q)^\dagger K_{\mathbf n}
M_{\mathbf n}(\mathbf q),\qquad
M_{\mathbf n}(\mathbf q)=A_{\mathbf n}-e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}I$$

which is manifestly Hermitian PSD. (An earlier version of this note claimed the
contribution collapses to $2K(1-\cos\mathbf q\cdot\mathbf R_{\mathbf n})$; it
does not — expanding leaves $B^{\mathsf T}KB$ and cross terms with
$B=A-I$. The $M^\dagger KM$ form above is the correct identity, and it makes
the conclusion immediate rather than approximate.)


In [ ]:
def skew(v):
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])

def contact_frame(R_n):
    """Orthonormal frame with z along R_n."""
    ez = R_n / np.linalg.norm(R_n)
    perp = np.array([1.,0.,0.]) if abs(ez[0]) < 0.9 else np.array([0.,1.,0.])
    ex = np.cross(perp, ez); ex /= np.linalg.norm(ex)
    return np.column_stack([ex, np.cross(ez, ex), ez])

def Gc_inv_matrix(c):
    """Inverse of the rigid-shift adjoint that moves a reference point by c."""
    G = np.eye(6); G[3:6, 0:3] = -skew(c)
    return G

def K_local_to_lab(K_local, R_frame, c_local):
    """K_local is expressed at the contact centroid, in local-frame axes.
    Returns K expressed at the far body's own centre of mass, in lab axes."""
    Gc_inv = Gc_inv_matrix(c_local)
    K_shifted = Gc_inv.T @ K_local @ Gc_inv
    Gamma = np.block([[R_frame, np.zeros((3,3))],[np.zeros((3,3)), R_frame]])
    return Gamma @ K_shifted @ Gamma.T

for c in shell_order:
    info = unique[c]
    R_frame = contact_frame(info['R_n'])
    d_c     = info['contact_pt'] - r_cm_at        # contact offset from body-0 COM
    info['R_frame'] = R_frame
    info['c_local'] = R_frame.T @ (info['R_n'] - d_c)
    info['Gamma']   = np.block([[R_frame, np.zeros((3,3))],
                                [np.zeros((3,3)), R_frame]])

TRIL_I, TRIL_J = np.tril_indices(6)
N_TRIL   = len(TRIL_I)                # 21
N_SHELL  = len(shell_order)
N_PARAMS = N_SHELL * N_TRIL

def theta_to_Lmap(theta):
    Lmap, idx = {}, 0
    for c in shell_order:
        L = np.zeros((6,6))
        L[TRIL_I, TRIL_J] = theta[idx:idx+N_TRIL]
        Lmap[c] = L
        idx += N_TRIL
    return Lmap

def Lmap_to_theta(Lmap):
    theta, idx = np.zeros(N_PARAMS), 0
    for c in shell_order:
        theta[idx:idx+N_TRIL] = Lmap[c][TRIL_I, TRIL_J]
        idx += N_TRIL
    return theta

def build_K(Lmap):
    """Cholesky factors -> (K in the local contact frame, K in the lab frame at
    each contact's far-body COM, the representation dynamical_matrix expects)."""
    K_local, K_lab = {}, {}
    for c in shell_order:
        Kl = Lmap[c] @ Lmap[c].T
        K_local[c] = Kl
        K_lab[c] = K_local_to_lab(Kl, unique[c]['R_frame'], unique[c]['c_local'])
    return K_local, K_lab

# --- Geometric prior: n isotropic point springs at the atom-pair midpoints ---
def contact_patch_stiffness(info, k_per_pair):
    d = info['midpoints'] - info['contact_pt']
    n = len(d)
    K = np.zeros((6,6))
    K[3:6, 3:6] = k_per_pair * n * np.eye(3)                        # translation
    K[0:3, 0:3] = k_per_pair * ((d**2).sum()*np.eye(3) - d.T @ d)   # rotation
    return K                                                        # T-R block = 0

def prior_Lmap(k_per_pair=PRIOR_K_PER_PAIR, ridge_frac=1e-3):
    Lmap, K_prior_local = {}, {}
    for c in shell_order:
        info = unique[c]
        Gamma = info['Gamma']
        Kl = Gamma.T @ contact_patch_stiffness(info, k_per_pair) @ Gamma
        Kl = 0.5*(Kl + Kl.T)
        Kl += ridge_frac * np.trace(Kl)/6 * np.eye(6)   # keep strictly PD
        K_prior_local[c] = Kl
        Lmap[c] = np.linalg.cholesky(Kl)
    return Lmap, K_prior_local

L_PRIOR, K_PRIOR_LOCAL = prior_Lmap()
_, K_LAB_PRIOR = build_K(L_PRIOR)

def regularizer_prior(K_local, lam=LAMBDA_PRIOR):
    """lam * Σ_n ||K_n - K_n^prior||_F² / ||K_n^prior||_F² -- scale-free per contact."""
    return lam*sum(np.sum((K_local[c]-K_PRIOR_LOCAL[c])**2)/np.sum(K_PRIOR_LOCAL[c]**2)
                   for c in shell_order)

def regularizer_prior_dK(K_local, lam=LAMBDA_PRIOR):
    return {c: 2*lam*(K_local[c]-K_PRIOR_LOCAL[c])/np.sum(K_PRIOR_LOCAL[c]**2)
            for c in shell_order}

print(f"Free parameters: {N_SHELL} contacts × {N_TRIL} Cholesky entries = {N_PARAMS}\n")
print(f"Geometric prior at k_per_pair = {PRIOR_K_PER_PAIR} k_BT/Å² "
      f"(= {PRIOR_K_PER_PAIR*0.414:.2f} N/m per atom pair):")
for c in shell_order:
    Kl = K_PRIOR_LOCAL[c]
    kt, kr = np.trace(Kl[3:6,3:6])/3, np.trace(Kl[0:3,0:3])/3
    print(f"  {str(c):14s} κ_T={kt:8.2f} k_BT/Å²   κ_R={kr:9.1f} k_BT/rad²   "
          f"ratio={kr/kt:6.1f} Å²   |T-R|={np.abs(Kl[0:3,3:6]).max():.2e}")
print("\n(The ratio column is the point: a flat isotropic guess sets it to 1.)")


## Dynamical Matrix

For contact $\mathbf n$ with stiffness $K_{\mathbf n}$ (expressed at the far
body's own centre of mass) and adjoint
$A_{\mathbf n}=\left(\begin{smallmatrix}I&0\\-[\mathbf R_{\mathbf n}]_\times&I
\end{smallmatrix}\right)$,

$$D_{\mathbf n}(\mathbf q)=M_{\mathbf n}(\mathbf q)^\dagger K_{\mathbf n}
M_{\mathbf n}(\mathbf q),\qquad
M_{\mathbf n}(\mathbf q)=A_{\mathbf n}-e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}I$$

and $D=\sum_{\mathbf n}D_{\mathbf n}$, summed over the 6 independent contacts
(both directions of each are already included). Building $D$ this way rather
than expanding into four terms makes it **exactly** Hermitian PSD in floating
point, so no cancellation error can push an acoustic eigenvalue negative.

$D(\mathbf q)$ is complex Hermitian, not real symmetric:
$\mathrm{Im}\,D_{\mathbf n}=\sin(\mathbf q\cdot\mathbf R_{\mathbf n})
\bigl(K_{\mathbf n}A_{\mathbf n}-(K_{\mathbf n}A_{\mathbf n})^{\mathsf T}\bigr)$,
generically $O(1)$. Every eigendecomposition below uses the full complex matrix.

**Zero modes at $\Gamma$.** $D(\mathbf q)u=0$ iff $M_{\mathbf n}(\mathbf q)u=0$
for every $\mathbf n$ (given $K_{\mathbf n}\succ0$). At $\mathbf q=0$ that reads
$\mathbf R_{\mathbf n}\times\boldsymbol\Omega=0$ for all $\mathbf n$, which for
three independent $\mathbf R_{\mathbf n}$ forces $\boldsymbol\Omega=0$ with
$\mathbf v$ free. So **exactly 3 eigenvalues vanish at $\Gamma$** — the pure
translations — and the 3 librational eigenvalues sit at whatever rest frequency
the rotational stiffness sets. A Born–von Kármán lattice with fixed
$\mathbf R_{\mathbf n}$ is not rotationally invariant, so a global rotation of
the crystal is not a zero mode of this model. That is a real limitation of the
model, not an error, but it is worth naming.


In [ ]:
def ad_translation(R_n):
    A = np.eye(6); A[3:6, 0:3] = -skew(R_n); return A

_I6 = np.eye(6)

def dynamical_matrix_batch(q_batch, unique, K_lab):
    """D(q) = Σ_n M_n(q)^† K_n M_n(q) over a batch of Cartesian q. (N,6,6) complex."""
    q_batch = np.atleast_2d(np.asarray(q_batch, float))
    D = np.zeros((len(q_batch), 6, 6), dtype=complex)
    for c, info in unique.items():
        R_n = info['R_n']; K = K_lab[c]
        A_n = ad_translation(R_n)
        phase = np.exp(1j*(q_batch @ R_n))                        # (N,)
        M = A_n[None,:,:] - phase[:,None,None]*_I6[None,:,:]      # (N,6,6)
        D += np.einsum('nji,jk,nkl->nil', M.conj(), K, M)
    return D

def dynamical_matrix(q_cart, unique, K_lab):
    return dynamical_matrix_batch(np.asarray(q_cart, float)[None,:], unique, K_lab)[0]

def mass_weighted_eigs(D_batch):
    """Eigenvalues of the pencil (D, M), via the Cholesky congruence. Complex
    Hermitian throughout -- taking .real here would be a different matrix."""
    Dw = np.einsum('ij,njk,lk->nil', Msq_inv, D_batch, Msq_inv)
    return np.linalg.eigvalsh(Dw)

# --- Sanity check at Gamma -------------------------------------------------
D0  = dynamical_matrix(np.zeros(3), unique, K_LAB_PRIOR)
ev0 = mass_weighted_eigs(D0[None])[0]
print("D(q=0) mass-weighted eigenvalues:", ev0.round(8))
print("  -> exactly 3 acoustic zeros expected; the upper 3 are the librational")
print("     rest frequencies and are NOT required to vanish (see markdown).")
assert np.sum(np.abs(ev0) < 1e-8*max(abs(ev0).max(), 1e-30)) == 3, \
    "expected exactly 3 zero modes at Gamma"

# --- Hermiticity / PSD check at a generic q --------------------------------
_qc = B_recip @ np.array([0.31, -0.17, 0.43])
_Dc = dynamical_matrix(_qc, unique, K_LAB_PRIOR)
print(f"\nAt a generic q: ||Im D|| / ||Re D|| = "
      f"{np.linalg.norm(_Dc.imag)/np.linalg.norm(_Dc.real):.4f}  (must be > 0)")
print(f"  Hermitian: {np.allclose(_Dc, _Dc.conj().T)}   "
      f"min eigenvalue: {np.linalg.eigvalsh(_Dc).min():.4e} (must be >= 0)")
print(f"  eigenvalues of Re(D) alone would be: {np.linalg.eigvalsh(_Dc.real)[:3].round(4)}")
print(f"  eigenvalues of the true complex D:   {np.linalg.eigvalsh(_Dc)[:3].round(4)}")
print("  -> these differ; using D.real is not an approximation but a different model.")


## Phonon Band Structure

Eigenvalues of the mass-weighted dynamical matrix along a path through the
Brillouin zone, using the geometric prior stiffness as the representative
starting model. Computed with a batched complex-Hermitian eigensolver: a few
hundred $6\times6$ problems per call, and this function is called once per frame
of the convergence movie later, so the serial version dominated runtime.

In [ ]:
HSP = {'Γ':np.array([0.,0.,0.]),'X':np.array([.5,0.,0.]),
       'Y':np.array([0.,.5,0.]),'Z':np.array([0.,0.,.5])}
path_labels = ['Γ','X','Γ','Y','Γ','Z','Γ']
N_seg = 60

q_frac, tick_idx = [], [0]
for seg in range(len(path_labels)-1):
    p0, p1 = HSP[path_labels[seg]], HSP[path_labels[seg+1]]
    last = (seg == len(path_labels)-2)
    for t in np.linspace(0, 1, N_seg, endpoint=last):
        q_frac.append(p0*(1-t)+p1*t)
    if not last: tick_idx.append(len(q_frac))
tick_idx.append(len(q_frac)-1)

q_frac = np.array(q_frac)
q_cart = (B_recip @ q_frac.T).T
x = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(q_cart,axis=0),axis=1))])

def bandstructure_freqs(K_lab, qs=None):
    """Batched and fully complex. ~50x faster than the per-q Python loop."""
    qs = q_cart if qs is None else qs
    ev = mass_weighted_eigs(dynamical_matrix_batch(qs, unique, K_lab))
    return np.sqrt(np.maximum(ev, 0)) * freq_unit

freqs_bs = bandstructure_freqs(K_LAB_PRIOR)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(x, freqs_bs, color='steelblue', lw=1.2)
for ti in tick_idx: ax.axvline(x[ti], color='k', lw=0.7)
ax.set_xticks([x[ti] for ti in tick_idx]); ax.set_xticklabels(path_labels)
ax.set_ylabel('Frequency (THz)'); ax.set_xlim(x[0],x[-1]); ax.set_ylim(0)
ax.set_title('Rigid-Body Phonon Band Structure — 6o2h (P1), geometric prior')
plt.tight_layout(); plt.savefig('band_structure.png', dpi=150); plt.show()

print(f"Acoustic branches vanish at Γ as required; librational rest frequencies "
      f"at Γ: {freqs_bs[0][3:].round(4)} THz")

## Diffuse Scattering Maps

$I(\mathbf q) = G^\dagger(\mathbf q)\,D^{-1}(\mathbf q)\,G(\mathbf q)$, with both
$G$ and $D$ complex.

$D(\mathbf q)$ is periodic in the reciprocal lattice, so $D(\mathbf G)=D(\mathbf 0)$
at every Bragg position: the 3 acoustic eigenvalues vanish there, and a
pseudo-inverse keeps $I(\mathbf q)$ finite. Immediately off a Bragg spot the
acoustic branches are linear in $|\delta\mathbf q|$ (their eigenvalues grow as
$|\delta\mathbf q|^2$), producing the $I\sim|\mathbf q-\mathbf G|^{-2}$ halos
visible on the log-scale maps.

Two things modulate halo brightness. It is *not* systematic absences — in $P1$
there are none, and the acoustic branches vanish at **every** reciprocal-lattice
point. It is $|G(\mathbf q)|^2\approx|\mathbf q F(\mathbf q)|^2$: halos are weak
wherever the molecular transform is weak, and grow overall as $|\mathbf q|^2$.

The elliptical boundary of the map comes from the spherical cutoff
$|\mathbf q|<2\pi/d_{\rm min}$ projected onto the $(h,k)$ plane, an ellipse
because $|\mathbf b_1|\neq|\mathbf b_2|$ for a triclinic cell.

In [ ]:
def diffuse_intensity_batch(K_lab, h_arr, k_arr, l_arr, chunk=None,
                            min_eig_frac=1e-6, G_arr=None):
    """I(q) = G^† D(q)^+ G, vectorized.

    FULL COMPLEX G and D. D is periodic in the reciprocal lattice, so D(G)=D(0)
    at every Bragg position: the 3 acoustic eigenvalues vanish there and a
    pseudo-inverse keeps I finite. Immediately off a Bragg spot the acoustic
    eigenvalues grow as |δq|², producing the I ~ |q-G|^-2 halos.
    """
    chunk = chunk or CHUNK_EVAL
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float)
    l_arr = np.asarray(l_arr, float)
    N = len(h_arr); I_out = np.zeros(N)
    for s in range(0, N, chunk):
        sl = slice(s, min(s+chunk, N))
        qb = (h_arr[sl,None]*B_recip[:,0] + k_arr[sl,None]*B_recip[:,1]
              + l_arr[sl,None]*B_recip[:,2])
        Db = dynamical_matrix_batch(qb, unique, K_lab)
        Gb = G_arr[sl] if G_arr is not None else G_at_hkl_batch(h_arr[sl], k_arr[sl], l_arr[sl])
        ev, evec = np.linalg.eigh(Db)                      # complex Hermitian
        emax = np.clip(ev.max(axis=1, keepdims=True), 1e-15, None)
        pos = ev > (min_eig_frac*emax)
        ev_inv = np.where(pos, 1.0/np.where(pos, ev, 1.0), 0.0)
        proj = np.einsum('nji,nj->ni', evec.conj(), Gb) * ev_inv
        DinvG = np.einsum('nij,nj->ni', evec, proj)
        I_out[sl] = np.real(np.einsum('ni,ni->n', Gb.conj(), DinvG))
    return I_out

def forward_I(K_lab, h, k, l, G=None):
    """Single-point I(q) by a plain (non-pseudo-inverse) solve -- used where
    D(q) is known to be comfortably invertible, i.e. away from Bragg points."""
    q = h*B_recip[:,0] + k*B_recip[:,1] + l*B_recip[:,2]
    if G is None:
        G = G_at_hkl(h, k, l)
    D = dynamical_matrix(q, unique, K_lab)
    return float(np.real(np.conj(G) @ np.linalg.solve(D, G)))

# CONSISTENCY CHECK -- the guard that a synthetic-only test cannot provide.
# The refinement objective uses forward_I's solve path; the maps and the ADP
# integral use diffuse_intensity_batch's eigendecomposition path. If those two
# ever disagree, one of them has silently dropped an imaginary part and the
# figures will be showing a different model from the one being fit.
_hq = np.array([1.37, -2.13, 0.61, 3.05, -1.44])
_kq = np.array([0.22,  1.81, -2.4, 0.13,  2.77])
_lq = np.array([0.45, -0.33, 1.12, -2.06, 0.88])
_Ia = diffuse_intensity_batch(K_LAB_PRIOR, _hq, _kq, _lq)
_Ib = np.array([forward_I(K_LAB_PRIOR, h, k, l) for h, k, l in zip(_hq, _kq, _lq)])
_rel = np.abs(_Ia - _Ib)/np.maximum(np.abs(_Ib), 1e-30)
print(f"Batched vs. single-point I(q): max relative difference {_rel.max():.3e}")
assert _rel.max() < 1e-8, "the two intensity code paths disagree -- see markdown"
print("OK: both intensity paths agree, so the plotted model is the fitted model.")


P = NPLOT_HK

def compute_diffuse_map(K_lab, L_layer=0):
    """I(h, k, L_layer) on the fine grid (multiples of 1/NPLOT_HK). Fully
    vectorized -- the old nested Python loop over (h,k) dominated runtime."""
    h_max = int(np.ceil(cell.a/D_MIN_DIFFUSE))+1
    k_max = int(np.ceil(cell.b/D_MIN_DIFFUSE))+1
    q_cut = 2*np.pi/D_MIN_DIFFUSE
    h_pts = np.arange(-h_max*P, h_max*P + 1)/P
    k_pts = np.arange(-k_max*P, k_max*P + 1)/P
    HH, KK = np.meshgrid(h_pts, k_pts, indexing='ij')
    hf, kf = HH.ravel(), KK.ravel()
    lf = np.full_like(hf, float(L_layer))
    q  = hf[:,None]*B_recip[:,0] + kf[:,None]*B_recip[:,1] + lf[:,None]*B_recip[:,2]
    qn = np.linalg.norm(q, axis=1)
    keep = (qn > 1e-3) & (qn < q_cut)
    print(f"Computing I(h,k,{L_layer}) at {keep.sum()} of {len(hf)} q-points "
          f"(Δh=Δk=1/{P}={1/P:.4f})...")
    I_flat = np.zeros(len(hf))
    I_flat[keep] = diffuse_intensity_batch(K_lab, hf[keep], kf[keep], lf[keep])
    return h_pts, k_pts, I_flat.reshape(HH.shape), h_max, k_max

def plot_diffuse(h_pts, k_pts, I_map, h_max, k_max, L_layer, fname):
    pos_vals = I_map[I_map > 0]
    if len(pos_vals) == 0:
        print("all I(q) = 0"); return
    vmax = np.percentile(pos_vals, 97)
    I_plot = np.clip(I_map, 0, vmax)
    fig, ax = plt.subplots(figsize=(7,6))
    im = ax.imshow((np.log1p(I_plot)/np.log(10)).T, origin='lower', cmap='inferno',
                   extent=[-h_max, h_max, -k_max, k_max], aspect='auto')
    plt.colorbar(im, ax=ax, label=r'$\log_{10}(1+I)$  [$k_BT$ units]')
    ax.set_xlabel('h'); ax.set_ylabel('k')
    ax.set_title(f'One-phonon diffuse scattering I(h,k,{L_layer}) — log scale\n'
                 f'NPLOT_HK={P}, D_min={D_MIN_DIFFUSE} Å')
    for hi in range(-h_max, h_max+1):
        for ki in range(-k_max, k_max+1):
            if hi == 0 and ki == 0: continue
            ax.plot(hi, ki, '+', color='white', alpha=0.25, markersize=4, markeredgewidth=0.5)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()
    print(f"Saved {fname}")

def plot_diffuse_cart(h_pts, k_pts, I_map, L_layer, fname_cart, n_grid=500):
    """Same map in Cartesian q-space (Å⁻¹), for direct comparison with an
    area-detector dataset."""
    H, K = np.meshgrid(h_pts, k_pts, indexing='ij')
    qx_all = H*B_recip[0,0] + K*B_recip[0,1] + L_layer*B_recip[0,2]
    qy_all = H*B_recip[1,0] + K*B_recip[1,1] + L_layer*B_recip[1,2]
    mask = I_map > 0
    qx, qy, I_vals = qx_all[mask], qy_all[mask], I_map[mask]
    if len(I_vals) < 10:
        print("Too few nonzero points for Cartesian map."); return
    vmax  = np.percentile(I_vals, 97)
    I_log = np.log1p(np.clip(I_vals, 0, vmax))/np.log(10)
    qx_lin = np.linspace(qx_all.min(), qx_all.max(), n_grid)
    qy_lin = np.linspace(qy_all.min(), qy_all.max(), n_grid)
    QX, QY = np.meshgrid(qx_lin, qy_lin)
    I_cart = griddata((qx, qy), I_log, (QX, QY), method='linear', fill_value=0.0)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(I_cart, origin='lower', cmap='inferno',
                   extent=[qx_lin[0], qx_lin[-1], qy_lin[0], qy_lin[-1]], aspect='equal')
    plt.colorbar(im, ax=ax, label=r'$\log_{10}(1+I)$  [$k_BT$ units]')
    ax.set_xlabel(r'$q_x$ (Å$^{-1}$)'); ax.set_ylabel(r'$q_y$ (Å$^{-1}$)')
    ax.set_title(f'Diffuse I(h,k,{L_layer}) — Cartesian $q$-space (Å$^{{-1}}$)')
    plt.tight_layout(); plt.savefig(fname_cart, dpi=150); plt.show()
    print(f"Saved {fname_cart}")

h_pts0, k_pts0, I_map0, h_max0, k_max0 = compute_diffuse_map(K_LAB_PRIOR, L_layer=0)
plot_diffuse(h_pts0, k_pts0, I_map0, h_max0, k_max0, 0, 'diffuse_map_hk0.png')
plot_diffuse_cart(h_pts0, k_pts0, I_map0, 0, 'diffuse_map_cart_hk0.png')

In [ ]:
h_pts5, k_pts5, I_map5, h_max5, k_max5 = compute_diffuse_map(K_LAB_PRIOR, L_layer=0.5)
plot_diffuse(h_pts5, k_pts5, I_map5, h_max5, k_max5, 0.5, 'diffuse_map_hk05.png')
plot_diffuse_cart(h_pts5, k_pts5, I_map5, 0.5, 'diffuse_map_cart_hk05.png')

## Synthetic Data

The "true" stiffness is the geometric prior (contact-patch springs scaled by the
number of atom pairs) **perturbed** by a sizeable random multiplicative
distortion in the Cholesky factor. Building the ground truth this way rather than
from a hand-picked diagonal matters: the prior is also the refinement's starting
point, so a ground truth drawn from a random perturbation of it tests whether the
data can move the fit a realistic distance, rather than testing recovery of a
target the prior was never near.

Diffuse intensities are evaluated at a stratified set of $(h,k,l)$ points: a
near-Bragg subset (dominated by the acoustic branches, sensitive to the
long-wavelength elastic limit) and a larger mid-zone subset (sensitive to the
short-range contact stiffnesses individually). Bragg positions themselves are
excluded. Independent Gaussian noise with a Poisson-like $\sigma\propto\sqrt I$
floor is added, and 15% of the points are held out for cross-validation.

In [ ]:
def true_Lmap(seed=0, distortion=0.35):
    """Ground truth = geometric prior × a random Cholesky-space distortion."""
    rng = np.random.default_rng(seed)
    Lmap = {}
    for c in shell_order:
        L0 = L_PRIOR[c]
        D_pert = np.tril(distortion*rng.standard_normal((6,6)))
        np.fill_diagonal(D_pert, 1.0 + distortion*rng.standard_normal(6))
        Lmap[c] = L0 @ D_pert
    return Lmap

L_TRUE = true_Lmap(seed=0)
K_LOCAL_TRUE, K_LAB_TRUE = build_K(L_TRUE)
print("True local-frame stiffness eigenvalues per contact (all > 0 required):")
for c in shell_order:
    ev_t = np.linalg.eigvalsh(K_LOCAL_TRUE[c])
    ev_p = np.linalg.eigvalsh(K_PRIOR_LOCAL[c])
    print(f"  {str(c):14s} true {ev_t.round(2)}")
    print(f"  {'':14s} prior{ev_p.round(2)}")
assert all(np.linalg.eigvalsh(K_LOCAL_TRUE[c]).min() > 0 for c in shell_order)

print("\n(The implied mean B of this true K is reported in the ADP section below,\n once bz_covariance is defined.)")

In [ ]:
def sample_q_points(n_near=60, n_mid=240, seed=1):
    rng = np.random.default_rng(seed)
    q_cut = 2*np.pi/D_MIN_DIFFUSE
    h_max = int(np.ceil(cell.a/D_MIN_DIFFUSE))+1
    k_max = int(np.ceil(cell.b/D_MIN_DIFFUSE))+1

    def snap(h, k):
        return round(h*P)/P, round(k*P)/P

    pts = []
    bragg_centers = [(h0,k0) for h0 in range(-3,4) for k0 in range(-3,4) if (h0,k0)!=(0,0)]
    rng.shuffle(bragg_centers)
    for (h0,k0) in bragg_centers:
        if len(pts) >= n_near: break
        for r in (0.06, 0.12, 0.2):
            ang = rng.uniform(0, 2*np.pi)
            h, k = snap(h0 + r*np.cos(ang), k0 + r*np.sin(ang))
            for l in (0.0, 0.5):
                q = h*B_recip[:,0] + k*B_recip[:,1] + l*B_recip[:,2]
                if 1e-3 < np.linalg.norm(q) < q_cut:
                    pts.append((h,k,l))
    pts = pts[:n_near]

    count = 0
    while count < n_mid:
        h, k = snap(rng.uniform(-h_max, h_max), rng.uniform(-k_max, k_max))
        l = rng.choice([0.0, 0.5])
        q = h*B_recip[:,0] + k*B_recip[:,1] + l*B_recip[:,2]
        qn = np.linalg.norm(q)
        if qn < 1e-3 or qn > q_cut: continue
        if min(abs(h-round(h)), abs(k-round(k))) < 0.05: continue   # skip near-Bragg
        pts.append((h,k,l)); count += 1
    return np.array(pts)

def well_conditioned_batch(K_lab, hkl, min_eig_frac=1e-4):
    """D(q) is exactly singular only at reciprocal-lattice points, but a point
    can still land near a degenerate direction, where the plain solve used by
    the objective is ill-conditioned. Screened at the TRUE K (i.e. at the
    operating point), and vectorized."""
    q = (hkl[:,0:1]*B_recip[:,0] + hkl[:,1:2]*B_recip[:,1] + hkl[:,2:3]*B_recip[:,2])
    ev = np.linalg.eigvalsh(dynamical_matrix_batch(q, unique, K_lab))
    emax, emin = ev[:,-1], ev[:,0]
    return (emax > 0) & (emin/np.where(emax > 0, emax, 1.0) > min_eig_frac)

q_all_raw = sample_q_points()
keep_mask = well_conditioned_batch(K_LAB_TRUE, q_all_raw)
q_all = q_all_raw[keep_mask]
print(f"{len(q_all)}/{len(q_all_raw)} sampled points kept after the conditioning "
      f"screen ({len(q_all_raw)-len(q_all)} discarded)")

G_all = G_at_hkl_batch(q_all[:,0], q_all[:,1], q_all[:,2])
I_true_all = diffuse_intensity_batch(K_LAB_TRUE, q_all[:,0], q_all[:,1], q_all[:,2],
                                     G_arr=G_all)

rng = np.random.default_rng(7)
NOISE_FLOOR = 0.02*np.median(I_true_all)
NOISE_SCALE = 0.06
sigma_all = NOISE_FLOOR + NOISE_SCALE*np.sqrt(np.clip(I_true_all, 0, None))
I_obs_all = I_true_all + rng.normal(0, sigma_all)

train_mask = rng.random(len(q_all)) > 0.15
q_train, G_train, I_train, sigma_train = (q_all[train_mask], G_all[train_mask],
                                          I_obs_all[train_mask], sigma_all[train_mask])
q_test,  G_test,  I_test,  sigma_test  = (q_all[~train_mask], G_all[~train_mask],
                                          I_obs_all[~train_mask], sigma_all[~train_mask])
w_train = 1.0/sigma_train**2

def effective_sample_size(w):
    """ESS = (Σw)²/Σw². Reported alongside the nominal count everywhere, because
    a nominal N means nothing once the weights are heterogeneous."""
    w = np.asarray(w, float)
    return w.sum()**2/np.sum(w**2)

print(f"{len(q_all)} q-points: {train_mask.sum()} training "
      f"(effective {effective_sample_size(w_train):.0f}), "
      f"{(~train_mask).sum()} held out for R_free")
print(f"{(I_obs_all < 0).mean():.1%} of synthetic observations are negative")

## Where the Fitted q-Points Sit

The sampled points span two Miller-index layers ($l=0$ and $l=0.5$), so they
are not coplanar in $\mathbf q$-space; the left panel shows all of them in
true Cartesian $\mathbf q$ (Å$^{-1}$), and the two right panels show each
layer separately, overlaid on that layer's true diffuse map, so the
near-Bragg and mid-zone sampling is visible relative to the halos.


In [ ]:
h_pts_bg0,  k_pts_bg0,  I_bg0,  h_max_bg0,  k_max_bg0  = compute_diffuse_map(K_LAB_TRUE, L_layer=0.0)
h_pts_bg05, k_pts_bg05, I_bg05, h_max_bg05, k_max_bg05 = compute_diffuse_map(K_LAB_TRUE, L_layer=0.5)

def q_cart_of(pts):
    return (pts[:,0:1]*B_recip[:,0] + pts[:,1:2]*B_recip[:,1] + pts[:,2:3]*B_recip[:,2])

qc_train, qc_test = q_cart_of(q_train), q_cart_of(q_test)

fig = plt.figure(figsize=(16, 5))
ax3d = fig.add_subplot(1, 3, 1, projection='3d')
ax3d.scatter(*qc_train.T, s=10, alpha=0.6, color='steelblue', label='training')
ax3d.scatter(*qc_test.T,  s=16, alpha=0.9, color='orangered', label='held out')
ax3d.set_xlabel('$q_x$'); ax3d.set_ylabel('$q_y$'); ax3d.set_zlabel('$q_z$')
ax3d.set_title('Sampled q-points (Å$^{-1}$)'); ax3d.legend(fontsize=8)

def overlay_panel(ax, h_pts, k_pts, I_bg, h_max, k_max, l_layer):
    pos = I_bg[I_bg > 0]
    vmax = np.percentile(pos, 97) if len(pos) else 1.0
    ax.imshow((np.log1p(np.clip(I_bg, 0, vmax))/np.log(10)).T, origin='lower', cmap='gray',
              extent=[-h_max, h_max, -k_max, k_max], aspect='auto', alpha=0.85)
    for pts, color, label in [(q_train, 'steelblue', 'training'), (q_test, 'orangered', 'held out')]:
        sel = np.abs(pts[:,2] - l_layer) < 1e-6
        ax.scatter(pts[sel,0], pts[sel,1], s=16, color=color, edgecolor='k',
                   linewidth=0.3, label=label)
    ax.set_xlabel('h'); ax.set_ylabel('k'); ax.set_title(f'l={l_layer}')
    ax.legend(fontsize=7, loc='upper right')

overlay_panel(fig.add_subplot(1,3,2), h_pts_bg0,  k_pts_bg0,  I_bg0,  h_max_bg0,  k_max_bg0,  0.0)
overlay_panel(fig.add_subplot(1,3,3), h_pts_bg05, k_pts_bg05, I_bg05, h_max_bg05, k_max_bg05, 0.5)

plt.tight_layout(); plt.savefig('sampled_q_points.png', dpi=150); plt.show()
print(f"{(q_all[:,2]==0.0).sum()} points at l=0, {(q_all[:,2]==0.5).sum()} points at l=0.5")

## Refining $K$ to the Synthetic Data

$$\mathcal L(\theta) = \sum_{\mathbf q\in\rm train} w(\mathbf q)
\bigl[I_{\rm obs}(\mathbf q)-I_{\rm model}(\mathbf q;\theta)\bigr]^2
+ \lambda\sum_{\mathbf n}
\frac{\bigl\|K_{\mathbf n}-K_{\mathbf n}^{\rm prior}\bigr\|_F^2}
     {\bigl\|K_{\mathbf n}^{\rm prior}\bigr\|_F^2}$$

**Regularizer.** The penalty pulls toward the geometric prior, not toward zero.
Penalizing only the local-frame $T$–$R$ block — the earlier choice — leaves
roughly 18 directions per contact, including every overall magnitude, with
nothing holding them; in a sloppy model those are precisely the directions that
drift to whichever boundary the optimizer reaches first. Each contact's term is
normalized by $\|K^{\rm prior}\|_F^2$ so one strong contact cannot dominate the
penalty purely by being stiff.

**Gradient.** Since $I=G^\dagger D^{-1}G$ and $d(D^{-1})=-D^{-1}dD\,D^{-1}$,
$dI=-v^\dagger dD\,v$ with $v=D^{-1}G$. $D$ is linear in each $K_{\mathbf n}$,
so $dD$ is a known linear function of $dK_{\mathbf n}$, giving closed-form
$dI/dK_{\mathbf n}$ at essentially no cost beyond the forward evaluation; the
chain rule through the fixed lab-frame transform and $K=LL^{\mathsf T}$ then
gives $d\mathcal L/dL$ (verified below against finite differences).

**Optimizer.** L-BFGS-B with the analytic gradient. The Cholesky parameterization
makes every point in parameter space a stable model, so the run is unconstrained.
The loss is fully vectorized over the point batch rather than looping over points
and contacts in Python. Convergence is *asserted*, not assumed: a run that stops
on `maxiter` has not converged and its parameters should not be quoted.

In [ ]:
def loss_and_grad(theta, q_list, G_list, I_obs, weights):
    """Vectorized over the whole point batch. G_list is precomputed once: it
    depends only on the structure and q, never on theta, so recomputing it inside
    a function L-BFGS-B calls every iteration would repeat the same sums
    hundreds of times."""
    Lmap = theta_to_Lmap(theta)
    K_local, K_lab = build_K(Lmap)
    N_mat = {c: Gc_inv_matrix(unique[c]['c_local']) @ unique[c]['Gamma'].T
             for c in shell_order}

    q_list = np.asarray(q_list, float); G_arr = np.asarray(G_list)
    q_batch = (q_list[:,0:1]*B_recip[:,0] + q_list[:,1:2]*B_recip[:,1]
               + q_list[:,2:3]*B_recip[:,2])
    D_batch = dynamical_matrix_batch(q_batch, unique, K_lab)
    # solve needs an explicit trailing RHS axis to batch over N systems
    V = np.linalg.solve(D_batch, G_arr[:,:,None])[:,:,0]
    Imodel = np.real(np.sum(np.conj(G_arr)*V, axis=1))
    resid = Imodel - np.asarray(I_obs)
    weights = np.asarray(weights)
    data_loss = float(np.sum(weights*resid**2))
    coeff = 2.0*weights*resid

    grad_L = {}
    for c in shell_order:
        R_n = unique[c]['R_n']
        A_n = ad_translation(R_n)
        phase = np.exp(1j*(q_batch @ R_n))
        U  = V @ A_n.T
        w1 = U - phase[:,None]*V
        w2 = V - phase.conj()[:,None]*U
        Graw = -(np.einsum('ni,nj->nij', U.conj(), w1) + np.einsum('ni,nj->nij', V.conj(), w2))
        G_A  = np.einsum('ij,njk,lk->nil', N_mat[c], Graw, N_mat[c])
        dL_c = np.real(np.einsum('nij,jk->nik', G_A + G_A.transpose(0,2,1), Lmap[c]))
        grad_L[c] = np.einsum('n,nij->ij', coeff, dL_c)

    reg  = regularizer_prior(K_local)
    dRdK = regularizer_prior_dK(K_local)
    for c in shell_order:
        grad_L[c] += (dRdK[c] + dRdK[c].T) @ Lmap[c]
        grad_L[c] = np.tril(grad_L[c])

    return data_loss + reg, Lmap_to_theta(grad_L)

In [ ]:
# Gradient check: analytic vs. central finite differences along random directions.
rng_chk = np.random.default_rng(42)
theta_chk = Lmap_to_theta(L_PRIOR)*(1 + 0.05*rng_chk.standard_normal(N_PARAMS))
n_chk = min(15, len(q_train))
q_chk, G_chk, I_chk, w_chk = q_train[:n_chk], G_train[:n_chk], I_train[:n_chk], w_train[:n_chk]

loss0, grad0 = loss_and_grad(theta_chk, q_chk, G_chk, I_chk, w_chk)
eps, errs = 1e-6, []
for idx in rng_chk.choice(N_PARAMS, size=8, replace=False):
    tp = theta_chk.copy(); tp[idx] += eps
    tm = theta_chk.copy(); tm[idx] -= eps
    fd = (loss_and_grad(tp, q_chk, G_chk, I_chk, w_chk)[0]
          - loss_and_grad(tm, q_chk, G_chk, I_chk, w_chk)[0])/(2*eps)
    errs.append(abs(fd - grad0[idx])/max(abs(fd), 1e-6))
print(f"Gradient check, max relative error over 8 random directions: {max(errs):.2e}")
assert max(errs) < 1e-4

In [ ]:
MAXITER = 5000

theta0 = Lmap_to_theta(L_PRIOR)          # start from the geometric prior
theta_history = [theta0.copy()]

result = minimize(loss_and_grad, theta0, args=(q_train, G_train, I_train, w_train),
                  jac=True, method='L-BFGS-B',
                  options={'maxiter': MAXITER, 'ftol': 1e-14, 'gtol': 1e-12},
                  callback=lambda xk: theta_history.append(xk.copy()))

print(result.message)
print(f"iterations: {result.nit}   final loss: {result.fun:.6g}   "
      f"||grad||: {np.linalg.norm(result.jac):.3e}")
if result.nit >= MAXITER:
    print("\n*** NOT CONVERGED: the run stopped on maxiter. Do not quote these "
          "parameters -- raise MAXITER or inspect the objective. ***")

theta_fit = result.x
_, K_LAB_FIT = build_K(theta_to_Lmap(theta_fit))

chi2_dof = float(np.sum(w_train*(np.array([forward_I(K_LAB_FIT, *q, G=G)
                                           for q, G in zip(q_train, G_train)]) - I_train)**2)
                 / max(len(q_train) - N_PARAMS, 1))
print(f"\nweighted χ²/dof (training) = {chi2_dof:.4f}")

## Watching the Band Structure Converge

The true band structure (from $K_{\rm true}$) is the thick grey line along the
same Γ–X–Γ–Y–Γ–Z–Γ path; the coloured line is the current Cholesky parameters at
each L-BFGS-B iteration. Only band structures are recomputed per frame (one
batched call, a few hundred $6\times6$ complex-Hermitian eigenproblems), never
the full diffuse map, and frames are rasterized and discarded one at a time.

Because the diffuse signal is dominated by the acoustic branches, expect the
low-frequency bands near Γ to lock onto the true curve quickly while the
librational branches — which contribute far less to $I(\mathbf q)$ — converge
more slowly. That asymmetry is the visible face of the sloppiness quantified
below: it is not a defect of the optimizer but a statement about which parameter
combinations the data actually constrains.

In [ ]:
freqs_true_bs = bandstructure_freqs(K_LAB_TRUE)
ymax = max(freqs_true_bs.max(), bandstructure_freqs(K_LAB_FIT).max())*1.15

frame_idx = np.unique(np.linspace(0, len(theta_history)-1,
                                  min(60, len(theta_history))).astype(int))

legend_handles = [Line2D([0],[0], color='0.6', lw=3.5, label='Ground truth'),
                  Line2D([0],[0], color='steelblue', lw=1.3, label='Current fit')]

images_bs = []
for fi in frame_idx:
    _, K_lab_fi = build_K(theta_to_Lmap(theta_history[fi]))
    freqs_fi = bandstructure_freqs(K_lab_fi)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x, freqs_true_bs, color='0.6', lw=3.5, zorder=1)
    ax.plot(x, freqs_fi, color='steelblue', lw=1.3, zorder=2)
    for ti in tick_idx: ax.axvline(x[ti], color='k', lw=0.7)
    ax.set_xticks([x[ti] for ti in tick_idx]); ax.set_xticklabels(path_labels)
    ax.set_ylabel('Frequency (THz)'); ax.set_xlim(x[0], x[-1]); ax.set_ylim(0, ymax)
    ax.set_title(f'Band-structure convergence — iteration {fi}/{len(theta_history)-1}')
    ax.legend(handles=legend_handles, loc='upper right', fontsize=8)
    plt.tight_layout()
    images_bs.append(fig_to_image(fig))
    plt.close(fig)

imageio.mimsave('band_structure_convergence.gif', images_bs, fps=8, loop=0)
del images_bs; gc.collect()
print(f"Saved band_structure_convergence.gif ({len(frame_idx)} frames from "
      f"{len(theta_history)} iterations)")

In [ ]:
def fit_stats(K_lab, q_list, G_list, I_list):
    """R-factor plus the linear correlation coefficient. R alone is a poor
    summary once observations can be negative -- Σ|I_obs| in the denominator is
    inflated by the magnitude of negative values, and R → 1 is also exactly what
    a model predicting ~0 produces, so R cannot distinguish 'poor fit' from
    'no fit'. CC separates those cases."""
    Imodel = diffuse_intensity_batch(K_lab, q_list[:,0], q_list[:,1], q_list[:,2],
                                     G_arr=np.asarray(G_list))
    R = np.sum(np.abs(Imodel - I_list))/np.sum(np.abs(I_list))
    CC = np.corrcoef(Imodel, I_list)[0,1]
    return R, CC, Imodel

R_train, CC_train, I_model_train = fit_stats(K_LAB_FIT, q_train, G_train, I_train)
R_free,  CC_free,  I_model_test  = fit_stats(K_LAB_FIT, q_test,  G_test,  I_test)
print(f"training set:  R = {R_train:.4f}   CC = {CC_train:.4f}")
print(f"held out:      R = {R_free:.4f}   CC = {CC_free:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10,4.5))
for ax, Im, Io, title in [(axes[0], I_model_train, I_train, 'training set'),
                          (axes[1], I_model_test,  I_test,  'held-out set ($R_{free}$)')]:
    ax.scatter(Io, Im, s=8, alpha=0.5)
    lim = [min(0, Io.min()*1.05), max(Io.max(), Im.max())*1.05]
    ax.plot(lim, lim, 'k--', lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('$I_{obs}$'); ax.set_ylabel('$I_{model}$'); ax.set_title(title)
plt.tight_layout(); plt.savefig('refinement_fit_quality.png', dpi=150); plt.show()

## How Many Parameters Does the Data Determine?

The model has 126 free parameters and no reason to believe the data constrains
126 independent combinations of them. The Gauss-Newton Hessian $J^{\mathsf T}J$,
built from the weighted Jacobian $J_{ij}=\sqrt{w_i}\,\partial I_i/\partial\theta_j$
at the optimum, answers this directly: its eigenvalue spectrum in a model of this
kind typically falls off close to geometrically over many decades — the
signature of a *sloppy* model in Sethna's sense.

What this buys, beyond a diagnosis:

* $n_{\rm eff}$, the number of parameter **combinations** the data pins down. The
  rest are set by the prior, and a fitted $K$ should be reported with that said
  out loud rather than quoted as 126 measured numbers.
* A principled value for `LAMBDA_PRIOR`: place it at the noise floor of the
  spectrum, where it controls everything the data does not and nothing it does.
* An explanation for the visible asymmetry in the convergence movie — the
  acoustic branches lock on quickly, the librational ones drift — since those are
  the stiff and sloppy directions respectively.

In [ ]:
# --- How many parameters does the data actually determine? -----------------
# The Sethna sloppiness diagnostic. Build the weighted Jacobian
# J_ij = sqrt(w_i) dI_i/dθ_j at the optimum and look at the spectrum of J^T J.
# For a model like this expect a handful of stiff directions out of 126, with
# the eigenvalues falling off geometrically over many decades. What this buys:
#   * n_eff -- the number of parameter COMBINATIONS the data constrains;
#   * a principled LAMBDA_PRIOR: place it at the noise floor of the spectrum,
#     so it controls everything the data does not and nothing it does;
#   * an honest way to report the result ("prior, plus data-determined
#     corrections along n_eff directions") instead of quoting 126 numbers as
#     though they were all measured.

def weighted_jacobian(theta, q_list, G_list, weights, scale=1.0, eps=1e-5):
    sw = np.sqrt(np.asarray(weights, float))
    q_list = np.asarray(q_list, float); G_arr = np.asarray(G_list)
    qb = (q_list[:,0:1]*B_recip[:,0] + q_list[:,1:2]*B_recip[:,1]
          + q_list[:,2:3]*B_recip[:,2])
    def I_of(th):
        _, K_lab = build_K(theta_to_Lmap(th))
        D = dynamical_matrix_batch(qb, unique, K_lab)
        V = np.linalg.solve(D, G_arr[:,:,None])[:,:,0]
        return scale*np.real(np.sum(np.conj(G_arr)*V, axis=1))
    Jm = np.zeros((len(q_list), len(theta)))
    for j in range(len(theta)):
        tp = theta.copy(); tp[j] += eps
        tm = theta.copy(); tm[j] -= eps
        Jm[:, j] = sw*(I_of(tp) - I_of(tm))/(2*eps)
    return Jm

def report_sloppiness(theta, q_list, G_list, weights, scale=1.0,
                      noise_floor_frac=1e-6, fname='sloppiness_spectrum.png'):
    Jm = weighted_jacobian(theta, q_list, G_list, weights, scale)
    ev = np.linalg.eigvalsh(Jm.T @ Jm)[::-1]
    ev = np.clip(ev, 1e-300, None)
    n_eff = int(np.sum(ev > noise_floor_frac*ev[0]))
    print(f"Gauss-Newton spectrum over {len(ev)} parameters:")
    print(f"  λ_max = {ev[0]:.4e}   λ_min = {ev[-1]:.4e}   "
          f"dynamic range = {ev[0]/ev[-1]:.2e}")
    print(f"  directions above {noise_floor_frac:g}·λ_max: {n_eff} / {len(ev)}")
    print(f"  -> the remaining {len(ev)-n_eff} directions are set by the prior, "
          f"not by the data. Say so when reporting K.")
    fig, ax = plt.subplots(figsize=(7,4))
    ax.semilogy(np.arange(1, len(ev)+1), ev/ev[0], 'o-', ms=3)
    ax.axhline(noise_floor_frac, color='r', ls='--', lw=1,
               label=f'noise floor ({noise_floor_frac:g})')
    ax.axvline(n_eff+0.5, color='k', ls=':', lw=1, label=f'$n_{{eff}}$ = {n_eff}')
    ax.set_xlabel('eigenvalue index'); ax.set_ylabel(r'$\lambda_i/\lambda_1$')
    ax.set_title('Sloppiness: Gauss-Newton eigenvalue spectrum')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()
    return ev, n_eff


In [ ]:
ev_spec, n_eff = report_sloppiness(theta_fit, q_train, G_train, w_train)

print(f"\nThe fit determines ~{n_eff} of {N_PARAMS} parameter combinations. "
      f"A reasonable LAMBDA_PRIOR sits near the spectrum's noise floor;")
print(f"  current LAMBDA_PRIOR = {LAMBDA_PRIOR:g}, "
      f"λ_max = {ev_spec[0]:.3e}, λ_max·1e-6 = {ev_spec[0]*1e-6:.3e}")

## Synthetic vs. Fitted Diffuse Scattering

The full diffuse map recomputed from the true and fitted stiffnesses, side by
side. The refinement only ever saw `I_train` at the sampled q-points; this map
compares the two models everywhere, including regions with no training data
nearby.

The two intensity panels are $\log_{10}(1+I)$; the two residual panels are
**not** log-transformed. The absolute-residual colourbar is literally
$I_{\rm fit}-I_{\rm true}$ in untransformed intensity units — and because $I$
spans many orders of magnitude between the acoustic halos and the weak mid-zone
signal, that residual is dominated by whatever happens at the brightest halo
pixels even when the fit is good everywhere else. The relative panel,
$(I_{\rm fit}-I_{\rm true})/I_{\rm true}$, divides out the local intensity scale
so good agreement in the faint mid-zone shows up instead of being swamped.

In [ ]:
h_pts_cmp, k_pts_cmp, I_true_map, h_max_cmp, k_max_cmp = h_pts_bg0, k_pts_bg0, I_bg0, h_max_bg0, k_max_bg0
_, _, I_fit_map, _, _ = compute_diffuse_map(K_LAB_FIT, L_layer=0)

def plot_diffuse_comparison(h_pts, k_pts, I_true, I_fit, h_max, k_max, fname, rel_clip=1.0):
    pos_vals = np.concatenate([I_true[I_true > 0], I_fit[I_fit > 0]])
    vmax = np.percentile(pos_vals, 97) if len(pos_vals) else 1.0
    resid = I_fit - I_true
    rmax = np.percentile(np.abs(resid), 99) if np.any(resid) else 1.0

    # Relative residual: divides out the intensity scale so the (very bright)
    # near-Bragg acoustic halos don't dominate the comparison the way they do
    # in the absolute residual. A small floor avoids dividing by ~0 near the
    # dark Bragg spots themselves.
    floor = 0.02 * vmax
    rel = resid / np.maximum(I_true, floor)
    rel_disp = np.clip(rel, -rel_clip, rel_clip)

    extent = [-h_max, h_max, -k_max, k_max]
    fig, axes = plt.subplots(2, 2, figsize=(11, 10))

    im0 = axes[0,0].imshow((np.log1p(np.clip(I_true, 0, vmax)) / np.log(10)).T, origin='lower',
                           cmap='inferno', extent=extent, aspect='auto')
    axes[0,0].set_title('Synthetic (true $K$)')
    plt.colorbar(im0, ax=axes[0,0], label=r'$\log_{10}(1+I)$')

    im1 = axes[0,1].imshow((np.log1p(np.clip(I_fit, 0, vmax)) / np.log(10)).T, origin='lower',
                           cmap='inferno', extent=extent, aspect='auto')
    axes[0,1].set_title('Fitted $K$')
    plt.colorbar(im1, ax=axes[0,1], label=r'$\log_{10}(1+I)$')

    im2 = axes[1,0].imshow(resid.T, origin='lower', cmap='RdBu_r',
                           extent=extent, aspect='auto', vmin=-rmax, vmax=rmax)
    axes[1,0].set_title('Absolute residual (fit − true)')
    plt.colorbar(im2, ax=axes[1,0], label='$\\Delta I$  (raw intensity units, not log)')

    im3 = axes[1,1].imshow(rel_disp.T, origin='lower', cmap='RdBu_r',
                           extent=extent, aspect='auto', vmin=-rel_clip, vmax=rel_clip)
    axes[1,1].set_title(f'Relative residual (fit − true)/true, clipped to ±{100*rel_clip:.0f}%')
    plt.colorbar(im3, ax=axes[1,1], label='$\\Delta I / I_{\\rm true}$')

    for ax in axes.flat:
        ax.set_xlabel('h'); ax.set_ylabel('k')
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

    rms_resid = np.sqrt(np.mean(resid**2))
    rms_true  = np.sqrt(np.mean(I_true**2))
    print(f"Saved {fname}")
    print(f"Absolute residual RMS: {rms_resid:.4g}  (I_true RMS: {rms_true:.4g})")
    print(f"Relative residual, unclipped — median |·|: {np.median(np.abs(rel)):.3g}, "
          f"90th pct |·|: {np.percentile(np.abs(rel), 90):.3g}")

plot_diffuse_comparison(h_pts_cmp, k_pts_cmp, I_true_map, I_fit_map, h_max_cmp, k_max_cmp,
                        'diffuse_map_true_vs_fit_hk0.png', 0.03)


## Predicted Atomic Displacement Parameters

The single-cell generalized-coordinate covariance
$\Sigma=\langle(\Omega,v)(\Omega,v)^{\mathsf T}\rangle$ follows from integrating
$D(\mathbf q)^{-1}$ over the Brillouin zone. Projecting onto an atom at
$\mathbf r$ via $\delta\mathbf r=\mathbf v+\boldsymbol\Omega\times(\mathbf r-\mathbf r_{\rm cm})$
gives that atom's full $3\times3$ anisotropic tensor $U=J_r\Sigma J_r^{\mathsf T}$
— the same object a crystallographic ANISOU record holds — with the isotropic
$B=\tfrac{8\pi^2}{3}\mathrm{Tr}(U)$ as its trace.

Both models are in the same reduced ($k_BT=1$) units, so an atom-by-atom
comparison of $B$ or the full $U$ is meaningful on an absolute scale. The
deposited structure's own ADPs are shown separately: comparing them to an
arbitrarily-chosen synthetic $K$ on an absolute scale is not meaningful — only
once $K$ is fit to real diffuse data does that comparison become direct.

**Two caveats that the synthetic comparison cannot surface**, and that matter the
moment this is run against real data:

1. $\Sigma$ is an integral of $D^{-1}$ whose acoustic part behaves as
   $\int d^3q/q^2$ — convergent in 3D, but easy to undersample. Grid convergence
   is checked below rather than assumed.
2. Comparing $\Sigma_{\rm true}$ against $\Sigma_{\rm fit}$ cancels any error
   *common to both*. A bug in `bz_covariance` itself is invisible here and shows
   up only against real ANISOU values. That is what the cross-path assertions
   earlier in the notebook are for.

In [ ]:
def bz_covariance(K_lab, n_grid=BZ_NGRID, offset=True, eig_floor=None, chunk=None):
    """Σ = <(Ω,v)(Ω,v)^T>, from integrating D(q)^-1 over the Brillouin zone.

    Three choices matter here and all three were wrong in an earlier version:

    * the FULL COMPLEX Hermitian D is used (D.real is a different matrix and
      shifts trace Σ by ~20% on this geometry);
    * the grid is OFFSET (Monkhorst-Pack style) so it never lands on Γ. The
      acoustic contribution goes as ∫d³q/q² -- convergent in 3D but badly
      undersampled by a coarse Γ-inclusive grid, and Σ is the headline ADP
      prediction, so this is not a detail;
    * the eigenvalue floor is ABSOLUTE, not relative to ev.max() at each q.
      With a strongly anisotropic K, a genuine small acoustic eigenvalue near Γ
      can otherwise be zeroed just because ev.max() happens to be large there.
    """
    chunk = chunk or CHUNK_EVAL
    g = (np.arange(n_grid) + (0.5 if offset else 0.0))/n_grid
    Ig, Jg, Kg = np.meshgrid(g, g, g, indexing='ij')
    q_batch = np.column_stack([Ig.ravel(), Jg.ravel(), Kg.ravel()]) @ B_recip.T
    N = len(q_batch)

    if eig_floor is None:
        Dref = dynamical_matrix_batch(q_batch[:min(N, 2000)], unique, K_lab)
        eig_floor = 1e-9 * mass_weighted_eigs(Dref).max()

    Sigma = np.zeros((6,6))
    for s0 in range(0, N, chunk):
        sl = slice(s0, min(s0+chunk, N))
        Db = dynamical_matrix_batch(q_batch[sl], unique, K_lab)
        Dw = np.einsum('ij,njk,lk->nil', Msq_inv, Db, Msq_inv)
        ev, evec = np.linalg.eigh(Dw)
        e_lab = np.einsum('ij,njk->nik', Msq_inv.T, evec)
        mask  = ev > eig_floor
        coeff = np.where(mask, 1.0/np.where(mask, ev, 1.0), 0.0)
        Sigma += np.real(np.einsum('ns,nis,njs->ij', coeff, e_lab, e_lab.conj()))
    return Sigma/N

def atom_adp_tensors(Sigma, positions):
    """Full anisotropic displacement tensor U (Å², 3x3) per atom. Vectorized."""
    d = np.asarray(positions) - r_cm_at
    N = len(d)
    Jr = np.zeros((N, 3, 6))
    Jr[:,0,1] =  d[:,2]; Jr[:,0,2] = -d[:,1]
    Jr[:,1,0] = -d[:,2]; Jr[:,1,2] =  d[:,0]
    Jr[:,2,0] =  d[:,1]; Jr[:,2,1] = -d[:,0]
    Jr[:,0,3] = Jr[:,1,4] = Jr[:,2,5] = 1.0
    return np.einsum('nia,ab,njb->nij', Jr, Sigma, Jr)

def isotropic_B(U):
    return (8*np.pi**2/3) * np.trace(U, axis1=1, axis2=2)

def mean_predicted_B(K_lab, n_grid=BZ_NGRID):
    return isotropic_B(atom_adp_tensors(bz_covariance(K_lab, n_grid=n_grid), apos)).mean()

def check_bz_convergence(K_lab, grids=(8, 12, 16, BZ_NGRID)):
    """Σ is the headline ADP prediction -- never report it without this."""
    print("BZ-grid convergence of the predicted mean B:")
    prev = None
    for n in grids:
        B = mean_predicted_B(K_lab, n_grid=n)
        delta = '' if prev is None else f'   Δ = {100*(B-prev)/prev:+.2f}%'
        print(f"  n_grid={n:3d}³ ({n**3:6d} q-points):  mean B = {B:8.3f} Å²{delta}")
        prev = B


Sigma_true = bz_covariance(K_LAB_TRUE)
Sigma_fit  = bz_covariance(K_LAB_FIT)
U_true, U_fit = atom_adp_tensors(Sigma_true, apos), atom_adp_tensors(Sigma_fit, apos)
B_true, B_fit = isotropic_B(U_true), isotropic_B(U_fit)

check_bz_convergence(K_LAB_TRUE)
print()
print(f"Mean predicted B (true K):  {B_true.mean():.3f} Å²")
print(f"Mean predicted B (fit K):   {B_fit.mean():.3f} Å²   "
      f"({100*(B_fit.mean()-B_true.mean())/B_true.mean():+.2f}%)")
print(f"Mean deposited B (6o2h):    {b_exp.mean():.2f} Å²  "
      f"(not comparable on an absolute scale to an arbitrary synthetic K)")

In [ ]:
# Per-residue means, compared directly (true vs. fitted, same units)
res_ids = []
for ch in st[0]:
    for res in ch:
        for atom in res:
            res_ids.append((ch.name, res.seqid.num))
res_index = {}
res_inverse = np.empty(len(res_ids), dtype=int)
for i, key in enumerate(res_ids):
    res_inverse[i] = res_index.setdefault(key, len(res_index))
n_res = len(res_index)
def per_residue_mean(x):
    out = np.zeros(n_res)
    counts = np.zeros(n_res)
    for v, r in zip(x, res_inverse):
        out[r] += v; counts[r] += 1
    return out / counts

B_true_res, B_fit_res = per_residue_mean(B_true), per_residue_mean(B_fit)

fig, axes = plt.subplots(1, 2, figsize=(11,4.5))

ax = axes[0]
ax.scatter(B_true, B_fit, s=6, alpha=0.4)
lim = [0, max(B_true.max(), B_fit.max())*1.05]
ax.plot(lim, lim, 'k--', lw=1)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('$B$ — true $K$ (Å²)'); ax.set_ylabel('$B$ — fitted $K$ (Å²)')
ax.set_title('Per-atom isotropic $B$: true vs. fitted')

ax = axes[1]
ax.plot(B_true_res, label='true $K$', lw=1.3)
ax.plot(B_fit_res, label='fitted $K$', lw=1.3, alpha=0.8)
ax.set_xlabel('residue index'); ax.set_ylabel('$B$ (Å²)')
ax.set_title('Per-residue mean $B$: true vs. fitted')
ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig('adp_comparison.png', dpi=150); plt.show()


### Anisotropic Comparison

The isotropic trace above discards the shape of each atom's displacement
ellipsoid. The full tensor comparison below checks whether the refinement
also recovers that shape: the six independent components of $U$ plotted
against each other (true vs. fitted, pooled over all atoms), and each
atom's anisotropy ratio (largest/smallest principal displacement) — a
model that only matches on the isotropic trace but gets the anisotropy
wrong would show good agreement in the $B$-factor plots above but poor
agreement here.


In [ ]:
def anisotropy_ratio(U):
    ev = np.linalg.eigvalsh(U)
    return ev[:, -1] / np.clip(ev[:, 0], 1e-12, None)

comp_idx   = [(0,0),(1,1),(2,2),(0,1),(0,2),(1,2)]
comp_names = ['$U_{11}$','$U_{22}$','$U_{33}$','$U_{12}$','$U_{13}$','$U_{23}$']

fig, axes = plt.subplots(1, 2, figsize=(11,4.5))

ax = axes[0]
for (i,j), name in zip(comp_idx, comp_names):
    ax.scatter(U_true[:,i,j], U_fit[:,i,j], s=5, alpha=0.4, label=name)
lim = [min(U_true.min(), U_fit.min()), max(U_true.max(), U_fit.max())]
ax.plot(lim, lim, 'k--', lw=1)
ax.set_xlabel('$U$ component — true $K$ (Å²)'); ax.set_ylabel('$U$ component — fitted $K$ (Å²)')
ax.set_title('Anisotropic tensor components: true vs. fitted')
ax.legend(fontsize=7, ncol=2)

ax = axes[1]
aniso_true, aniso_fit = anisotropy_ratio(U_true), anisotropy_ratio(U_fit)
ax.scatter(aniso_true, aniso_fit, s=6, alpha=0.4)
lim2 = [1, max(aniso_true.max(), aniso_fit.max())*1.05]
ax.plot(lim2, lim2, 'k--', lw=1)
ax.set_xlabel('anisotropy ratio — true $K$'); ax.set_ylabel('anisotropy ratio — fitted $K$')
ax.set_title('Displacement-ellipsoid anisotropy: true vs. fitted')

plt.tight_layout(); plt.savefig('adp_anisotropic_comparison.png', dpi=150); plt.show()


In [ ]:
# Deposited ANISOU comparison, when present: this is the comparison that
# becomes meaningful once K is fit to real (not synthetic) diffuse data.
if has_aniso.mean() > 0.5:
    fig, ax = plt.subplots(figsize=(5,5))
    for (i,j), name in zip(comp_idx, comp_names):
        ax.scatter(u_exp[has_aniso,i,j], U_fit[has_aniso,i,j], s=5, alpha=0.4, label=name)
    lim = [min(u_exp[has_aniso].min(), U_fit[has_aniso].min()),
           max(u_exp[has_aniso].max(), U_fit[has_aniso].max())]
    ax.plot(lim, lim, 'k--', lw=1)
    ax.set_xlabel('$U$ component — deposited 6o2h (Å²)')
    ax.set_ylabel('$U$ component — fitted $K$ (Å²)')
    ax.set_title('Anisotropic $U$: fitted model vs. deposited (absolute scale\n'
                 'not meaningful until $K$ is fit to real data)')
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout(); plt.savefig('adp_vs_deposited_anisotropic.png', dpi=150); plt.show()
else:
    print(f"Only {has_aniso.mean():.1%} of atoms in 6o2h carry ANISOU records "
          f"(full anisotropic refinement typically needs atomic resolution); "
          f"skipping the deposited anisotropic comparison. Isotropic B-factors "
          f"are compared above regardless.")


## Phonon Mode Visualization

Each phonon mode at a chosen $\mathbf q$-point is rendered as a looping GIF
of the rigid-body motion, using the fitted model. $|\Omega|$ and $|v|$ are
the rotational and translational norms of the mass-weighted eigenvector
(normalized so $\Omega^{\mathsf T}J\Omega+v^{\mathsf T}mv=1$); their ratio
gives the rotational kinetic-energy fraction, used below to label each mode
librational, translational, or mixed. Displacement amplitude is defined as
the largest per-atom displacement in Å, not the center-of-mass displacement.


In [ ]:
Q_VIZ     = np.array([0.5, 0.25, 0.0])  # fractional — zone-boundary, off-axis
AMPLITUDE = 3.0    # Å — maximum per-atom displacement
N_FRAMES  = 30
GIF_FPS   = 12

q_viz  = B_recip @ Q_VIZ
D_viz  = dynamical_matrix(q_viz, unique, K_LAB_FIT)
# Complex Hermitian, as everywhere else -- Re(D) has different eigenvalues, and
# the acoustic (smallest) ones are the worst affected, so a .real here would
# animate a different set of modes from the ones that were fit.
Dw_viz = np.einsum('ij,jk,lk->il', Msq_inv, D_viz, Msq_inv)
ev_v, evec_v = np.linalg.eigh(Dw_viz)
ev_v      = np.maximum(ev_v, 0)
freqs_viz = np.sqrt(ev_v) * freq_unit
evecs_lab = Msq_inv.T @ evec_v          # complex, one column per mode

# A phonon eigenvector at general q is genuinely complex: the physical motion is
# Re(e e^{i(q·R - ωt)}), so different components lead each other in phase. For a
# single-cell animation we rotate each mode by the global phase that makes it as
# real as possible (which is exact for a standing mode and a good approximation
# otherwise) and record how much amplitude the residual imaginary part carries.
_phase = np.exp(-1j*np.angle(evecs_lab[np.abs(evecs_lab).argmax(axis=0),
                                       np.arange(6)]))
evecs_lab = evecs_lab * _phase[None, :]
_im_frac = np.linalg.norm(evecs_lab.imag, axis=0)/np.linalg.norm(evecs_lab, axis=0)
print(f"Residual out-of-phase amplitude per mode: {_im_frac.round(3)}")
print("  (0 = a pure standing mode the single-cell animation represents exactly;")
print("   large values mean the components genuinely lead each other in phase.)")

print(f"Modes at q={Q_VIZ}:")
hdr = f"{'Mode':>5} {'Freq (THz)':>14} {'|Ω|':>9} {'|v|':>9}  {'rot KE %':>9}  character"
print(hdr); print('-'*len(hdr))
for s in range(6):
    Om = evecs_lab[:3, s].real; v = evecs_lab[3:, s].real
    rot_KE, trans_KE = float(Om @ J @ Om), float(m_total * (v@v))
    rot_pct = 100 * rot_KE / max(rot_KE + trans_KE, 1e-30)
    char = 'librational' if rot_pct > 60 else ('translational' if rot_pct < 40 else 'mixed')
    print(f"  {s:3d}  {freqs_viz[s]:14.6f}  {np.linalg.norm(Om):9.4f}  "
          f"{np.linalg.norm(v):9.4f}  {rot_pct:9.1f}%  {char}")


In [ ]:
# Molecular isosurface: marching cubes on the calculated model density
# (rho_model, built earlier for visualization only -- the molecular transform
# used for G is summed over atoms, not read off this grid), downsampled
# for speed, keeping only the largest connected component so periodic-image
# and solvent-void fragments are discarded, then Lambert-shaded per face so
# shape and roughness stay visible even at partial transparency.
base_pos = apos.copy()
pts_cm   = base_pos - r_cm_at

MC_DS = 0.5
rho_mc = nd_zoom(rho_model.astype(np.float32), MC_DS, order=1)
nu_mc, nv_mc, nw_mc = rho_mc.shape
_iso_level = rho_mc.max() * 0.05
mc_verts_grid, mc_faces_raw, _normals, _vals = marching_cubes(rho_mc, level=_iso_level)
_frac = mc_verts_grid / np.array([nu_mc, nv_mc, nw_mc])
mc_verts_raw = (A_orth @ _frac.T).T

def _largest_component(verts, faces):
    n = len(verts)
    idx_i = np.concatenate([faces[:,0], faces[:,1], faces[:,2]])
    idx_j = np.concatenate([faces[:,1], faces[:,2], faces[:,0]])
    adj   = csr_matrix((np.ones(len(idx_i), dtype=np.int8), (idx_i, idx_j)), shape=(n, n))
    _, labels = sc_connected_components(adj, directed=False)
    keep_label = np.bincount(labels).argmax()
    keep = np.where(labels == keep_label)[0]
    remap = np.full(n, -1, dtype=int); remap[keep] = np.arange(len(keep))
    face_ok = (labels[faces] == keep_label).all(axis=1)
    return verts[keep], remap[faces[face_ok]]

mc_verts_cart, mc_faces = _largest_component(mc_verts_raw, mc_faces_raw)
mc_verts_cm = mc_verts_cart - r_cm_at
print(f"Isosurface: {len(mc_verts_cm)} verts, {len(mc_faces)} tris "
      f"(largest connected component, level={_iso_level:.3f} e/Å³)")

_LIGHT = np.array([0.5, 0.8, 1.0]); _LIGHT /= np.linalg.norm(_LIGHT)

def _shade(tri_array, hex_color, alpha, ambient=0.35):
    v0, v1, v2 = tri_array[:,0], tri_array[:,1], tri_array[:,2]
    n = np.cross(v1 - v0, v2 - v0)
    mag = np.linalg.norm(n, axis=1, keepdims=True)
    n /= np.where(mag > 1e-12, mag, 1.0)
    intensity = ambient + (1 - ambient) * np.abs(n @ _LIGHT)
    r = int(hex_color[1:3], 16) / 255
    g = int(hex_color[3:5], 16) / 255
    b = int(hex_color[5:7], 16) / 255
    return np.column_stack([intensity*r, intensity*g, intensity*b, np.full(len(tri_array), alpha)])

def displaced_mc(mode_idx, scale):
    Om_raw = evecs_lab[:3, mode_idx].real.copy()
    v_raw  = evecs_lab[3:, mode_idx].real.copy()
    norm_vec = np.sqrt(Om_raw @ Om_raw + v_raw @ v_raw)
    if norm_vec < 1e-10:
        return mc_verts_cm + r_cm_at
    Om_u, v_u = Om_raw/norm_vec, v_raw/norm_vec
    delta_unit = v_u[None,:] + np.cross(Om_u[None,:], pts_cm)
    max_d = np.linalg.norm(delta_unit, axis=1).max()
    if max_d < 1e-12:
        return mc_verts_cm + r_cm_at
    fac = scale / max_d
    Om_s, v_s = fac*Om_u, fac*v_u
    if abs(fac)*np.linalg.norm(Om_u) > 1e-10:
        return Rot.from_rotvec(Om_s).apply(mc_verts_cm) + r_cm_at + v_s
    return mc_verts_cm + r_cm_at + v_s

eq_bbox_min = mc_verts_cart.min(axis=0)
eq_bbox_max = mc_verts_cart.max(axis=0)


In [ ]:
MODE_COLORS = ['#4e9de0', '#e05c5c', '#4fba74', '#e0b14e', '#a56be0', '#e07e4e']
SURF_ALPHA  = 0.65
ELEV, AZIM  = 20, -60

def make_mode_gif(mode_idx, amplitude=AMPLITUDE, n_frames=N_FRAMES, fps=GIF_FPS, fname=None, figsize=(6,5)):
    """Render one phonon mode as a looping GIF and save it to disk (no
    in-notebook display, and memory is freed after each mode)."""
    if fname is None:
        qstr = '_'.join(f'{x:.2f}' for x in Q_VIZ)
        fname = f'mode_{mode_idx}_q{qstr}.gif'
    color  = MODE_COLORS[mode_idx % len(MODE_COLORS)]
    scales = amplitude * np.sin(2*np.pi*np.arange(n_frames)/n_frames)
    pad = amplitude * 2.5
    xlim = (eq_bbox_min[0]-pad, eq_bbox_max[0]+pad)
    ylim = (eq_bbox_min[1]-pad, eq_bbox_max[1]+pad)
    zlim = (eq_bbox_min[2]-pad, eq_bbox_max[2]+pad)

    Om_r = evecs_lab[:3, mode_idx].real
    rot_KE   = float(Om_r @ J @ Om_r)
    trans_KE = m_total * float(np.linalg.norm(evecs_lab[3:, mode_idx].real)**2)
    rot_pct  = 100*rot_KE/(rot_KE+trans_KE+1e-30)

    images = []
    for sc in scales:
        fig = plt.figure(figsize=figsize, facecolor='#0d1117')
        ax  = fig.add_subplot(111, projection='3d', facecolor='#0d1117')
        try:
            verts = displaced_mc(mode_idx, sc)
            tris  = verts[mc_faces]
            rgba  = _shade(tris, color, SURF_ALPHA)
            poly  = Poly3DCollection(tris, facecolors=rgba, linewidths=0, edgecolor='none')
            ax.add_collection3d(poly)
            ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
            ax.set_axis_off(); ax.view_init(elev=ELEV, azim=AZIM)
            ax.set_title(f'Mode {mode_idx}   {freqs_viz[mode_idx]:.4f} THz\n'
                         f'q={list(Q_VIZ)}   rot {rot_pct:.0f}%  trans {100-rot_pct:.0f}%',
                         color='white', fontsize=8.5, pad=3)
            plt.tight_layout(pad=0.2)
            images.append(fig_to_image(fig))
        finally:
            plt.close(fig)

    imageio.mimsave(fname, images, fps=fps, loop=0)
    del images; gc.collect()
    return fname

print(f"Generating surface GIFs at q={Q_VIZ} (amplitude={AMPLITUDE} Å max per-atom)...")
for s in range(6):
    print(f"Mode {s}: {freqs_viz[s]:.6f} THz", end="  ... ")
    try:
        path = make_mode_gif(s)
        print(f"saved -> {path}")
    except Exception as e:
        print(f"FAILED: {e}")
    gc.collect()
print("Load any GIF with: display(IPImage(filename='mode_N_q....gif'))")


In [ ]:
# Supercell wave: each cell (i,j,k) is a rigid copy displaced by
# amplitude * cos(2*pi*q_frac.[i,j,k] + phase). The central cell is
# colored distinctly from its neighbors so the propagating wave pattern
# can be read off against a fixed reference point.
SUPER_N1, SUPER_N2, SUPER_N3 = 3, 3, 1   # a full 3x3x3 renders 27 isosurfaces per
                                          # frame and can exhaust notebook-kernel memory;
                                          # 3x3x1 already shows the in-plane wave pattern
MODE_SUPER   = 0
SUPER_COLOR  = '#4e9de0'
SUPER_ALPHA  = 0.38
CENTER_COLOR = '#e05c5c'
CENTER_ALPHA = 0.55

def make_supercell_gif(mode_idx=MODE_SUPER, amplitude=AMPLITUDE, n_frames=N_FRAMES, fps=GIF_FPS,
                       n1=SUPER_N1, n2=SUPER_N2, n3=SUPER_N3, fname=None, figsize=(10,9)):
    if fname is None:
        qstr = '_'.join(f'{x:.2f}' for x in Q_VIZ)
        fname = f'supercell_mode{mode_idx}_q{qstr}.gif'

    cells_ijk = [(i,j,k) for i in range(n1) for j in range(n2) for k in range(n3)]
    center_ijk = (n1//2, n2//2, n3//2)
    phase0 = np.array([2*np.pi*(Q_VIZ[0]*i + Q_VIZ[1]*j + Q_VIZ[2]*k) for i,j,k in cells_ijk])
    T_cell = np.array([i*a1 + j*a2 + k*a3 for i,j,k in cells_ijk])

    all_eq = np.vstack([mc_verts_cart + T for T in T_cell])
    pad = amplitude * 2.5
    xlim = (all_eq[:,0].min()-pad, all_eq[:,0].max()+pad)
    ylim = (all_eq[:,1].min()-pad, all_eq[:,1].max()+pad)
    zlim = (all_eq[:,2].min()-pad, all_eq[:,2].max()+pad)

    images = []
    for fi in range(n_frames):
        t = 2*np.pi*fi/n_frames
        fig = plt.figure(figsize=figsize, facecolor='#0d1117')
        ax  = fig.add_subplot(111, projection='3d', facecolor='#0d1117')
        try:
            for (i,j,k), T, ph0 in zip(cells_ijk, T_cell, phase0):
                sc = amplitude * np.cos(ph0 + t)
                verts = displaced_mc(mode_idx, sc) + T
                tris_v = verts[mc_faces]
                is_center = (i,j,k) == center_ijk
                color = CENTER_COLOR if is_center else SUPER_COLOR
                alpha = CENTER_ALPHA if is_center else SUPER_ALPHA
                rgba = _shade(tris_v, color, alpha)
                poly = Poly3DCollection(tris_v, facecolors=rgba, linewidths=0, edgecolor='none')
                ax.add_collection3d(poly)

            ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
            ax.set_axis_off(); ax.view_init(elev=25, azim=-55)
            ax.set_title(f'Supercell phonon — Mode {mode_idx}  {freqs_viz[mode_idx]:.4f} THz\n'
                         f'q={list(Q_VIZ)}   {n1}×{n2}×{n3} cells (center cell highlighted)',
                         color='white', fontsize=8.5, pad=3)
            plt.tight_layout(pad=0.2)
            images.append(fig_to_image(fig))
        finally:
            plt.close(fig)

    imageio.mimsave(fname, images, fps=fps, loop=0)
    del images; gc.collect()
    return fname

print(f"Generating {SUPER_N1}×{SUPER_N2}×{SUPER_N3} supercell GIF for mode {MODE_SUPER}...")
try:
    sc_path = make_supercell_gif()
    print(f"Saved -> {sc_path}")
except Exception as e:
    print(f"Supercell GIF failed: {e}")
    import traceback; traceback.print_exc()


## Summary

| Output | File | Description |
|--------|------|-------------|
| Band structure | `band_structure.png` | Phonon dispersion Γ–X–Y–Z–Γ (geometric prior) |
| Diffuse map l=0 | `diffuse_map_hk0.png`, `diffuse_map_cart_hk0.png` | TDS in the hk0 plane |
| Diffuse map l=0.5 | `diffuse_map_hk05.png`, `diffuse_map_cart_hk05.png` | TDS at half-integer l |
| Refinement fit quality | `refinement_fit_quality.png` | $I_{\rm obs}$ vs $I_{\rm model}$, training and held-out |
| Sampled q-points | `sampled_q_points.png` | 3D Cartesian view plus per-layer overlays |
| Band-structure convergence | `band_structure_convergence.gif` | True vs. fit, frame per L-BFGS-B iteration |
| Sloppiness spectrum | `sloppiness_spectrum.png` | Gauss-Newton eigenvalues; how many parameters the data determines |
| Synthetic vs. fitted map | `diffuse_map_true_vs_fit_hk0.png` | True $K$, fitted $K$, and residuals |
| ADP comparison (isotropic) | `adp_comparison.png` | Per-atom and per-residue $B$: true vs. fitted |
| ADP comparison (anisotropic) | `adp_anisotropic_comparison.png` | Full $U$ components and anisotropy ratio |
| ADP vs. deposited | `adp_vs_deposited_anisotropic.png` | Fitted model vs. 6o2h ANISOU |
| Mode GIFs | `mode_N_q*.gif` | All 6 modes at the chosen q-point |
| Supercell wave | `supercell_mode*.gif` | 3×3×1 supercell phonon wave |

**Molecular transform.** $F$ and $L$ are summed directly over the deposited
atomic coordinates with IT92 form factors, referenced to the atomic centre of
mass — the same point the dynamical matrix uses. A periodic density grid would
give the wrong $F(\mathbf q)$ at the non-integer $(h,k,l)$ this pipeline runs on,
because a molecule that crosses a cell boundary contributes with unequal phase
factors. The measured Bragg amplitudes enter as a smooth resolution-dependent
amplitude correction instead of through a hybrid density.

**Contact stiffness parameterization.** Each of the 6 distinct contacts (distinct,
not symmetry-distinct: $P1$ has no point symmetry) carries an independent
$6\times6$ stiffness, specified in a local frame anchored at the measured contact
centroid via a Cholesky factor $K=LL^{\mathsf T}$. The prior is the stiffness of
$n$ isotropic point springs at the atom-pair midpoints, which supplies the
contact-count scaling, the $\kappa_R/\kappa_T\sim\rho_g^2$ ratio, and the patch
anisotropy for free. Refinement is regularized toward that prior.

**Everything is complex.** $I=G^\dagger D^{-1}G$ has complex $G$ and complex
Hermitian $D$; $\mathrm{Im}\,D_{\mathbf n}=\sin(\mathbf q\cdot\mathbf R_{\mathbf n})
(K A-(KA)^{\mathsf T})$ is generically $O(1)$. Replacing either by its real part
is not an approximation but a different model, and the two intensity code paths
are asserted against each other so the plotted model is always the fitted model.

**Refinement.** L-BFGS-B on the Cholesky parameters with an analytic gradient
verified against finite differences, fully vectorized over the point batch.
Convergence is asserted rather than assumed. Training and held-out $R$ **and**
$CC$ are reported, and the Gauss-Newton spectrum states how many of the 126
parameters the data actually determines.